In [3]:
import os
import requests
import time

from dotenv import load_dotenv

In [4]:
# Loading the .env file from one folder above notebooks.
# Grab the value stored in STATS_API_KEY
load_dotenv("../.env")

stats_api_key = os.getenv("STATS_API_KEY")

print("TheStatsAPI key loaded:", stats_api_key is not None)

TheStatsAPI key loaded: True


In [5]:
# Instead of repeating the url, we store it as BASE_URL
BASE_URL = "https://api.thestatsapi.com/api/football"

# Our version of authentication.
headers = {
    "Authorization": f"Bearer {stats_api_key}"
}

In [6]:
# We're packaging what we did yesterday into one reusable function
# Later we will be able to request and the function will handle everything and convert JSON response.
def get_api_data(url, max_retries = 5):
    wait_time = 5

    for attempt in range(max_retries):
        try:
            response = requests.get(
                url,
                headers = headers,
                timeout = 30
            )

        except requests.exceptions.RequestException as e:
            print(
                f"Connection error: {e}."
                f"Waiting {wait_time} seconds before retry "
                f"{attempt + 1}/{max_retries}..."
            )

        if response.status_code == 200:
            return response.json()

        if response.status_code == 429:
            print(
                f"Rate limit reached. "
                f"Waiting {wait_time} seconds before retry "
                f"{attempt + 1}/{max_retries}"
            )

            time.sleep(wait_time)
            wait_time *= 2
            continue

        print("Request failed:", response.status_code)
        print(response.text[:500])
        return None

    print("Request failed after maximum retries.")
    return None


# This whole process is called exponential backoff. 
# We don't want to keep retrying because this will lead to incomplete data.
# We want each match to be saved in one single try, not multiple retries.

In [7]:
# This function takes a match ID and builds the correct endpoint URL.
# The URL is then sent through the get_api_data function we created.
def get_match_details(match_id):
    url = f"{BASE_URL}/matches/{match_id}"
    return get_api_data(url)

In [8]:
match_id = "mt_511976372"

# Takes a match ID and returns a clean match record.
match_details = get_match_details(match_id)

print(match_details)

{'data': {'id': 'mt_511976372', 'competition_id': 'comp_3039', 'competition_name': 'Premier League', 'season_id': 'sn_8406098', 'matchday': 1, 'stage_name': None, 'group_label': None, 'status': 'finished', 'utc_date': '2026-08-21T19:00:00.000Z', 'home_team': {'id': 'tm_9145', 'name': 'Arsenal'}, 'away_team': {'id': 'tm_2954', 'name': 'Coventry City'}, 'score': {'home': 3, 'away': 0, 'final_score': None, 'regulation': {'home': 3, 'away': 0}, 'after_extra_time': None, 'penalty_shootout': None, 'went_to_extra_time': False, 'went_to_penalties': False, 'winner': 'home', 'half_time_home': 2, 'half_time_away': 0}, 'venue': {'name': 'Emirates Stadium', 'city': 'London'}, 'is_neutral': False, 'referee': {'id': 'ref_72946204', 'name': 'Thomas Bramall'}, 'odds_available': True, 'live_odds_available': False, 'xg_available': True}}


In [9]:
# Let's clean up our output so it's not just one messy Python dictionary.
match = match_details["data"]

print("Competition:", match["competition_name"])
print("Matchday:", match["matchday"])
print("Date:", match["utc_date"])
print("Home team:", match["home_team"]["name"])
print("Away team:", match["away_team"]["name"])
print("Score:", match["score"]["home"], "-", match["score"]["away"])
print("Venue:", match["venue"]["name"])
print("Referee:", match["referee"]["name"])
print("xG available:", match["xg_available"])
print("Odds available:", match["odds_available"])


Competition: Premier League
Matchday: 1
Date: 2026-08-21T19:00:00.000Z
Home team: Arsenal
Away team: Coventry City
Score: 3 - 0
Venue: Emirates Stadium
Referee: Thomas Bramall
xG available: True
Odds available: True


In [10]:
# Creating a function that retrieves statistics given a specific Match ID.
def get_match_statistics(match_id):
    url = f"{BASE_URL}/matches/{match_id}/stats"
    return get_api_data(url)

In [11]:
match_statistics = get_match_statistics(match_id)
print(match_statistics.keys())

dict_keys(['data'])


In [12]:
# Cleaning up our match statistics
# Going from a messy Python dictionary to a clean format.

stats = match_statistics["data"]

print("Possession:", stats["overview"]["ball_possession"]["all"])
print("xG:", stats["overview"]["expected_goals"]["all"])
print("Shots:", stats["overview"]["total_shots"]["all"])
print("Shots on target:", stats["overview"]["shots_on_target"]["all"])
print("Passes:", stats["overview"]["passes"]["all"])
print("Corners:", stats["overview"]["corner_kicks"]["all"])

Possession: {'home': 64, 'away': 36}
xG: {'home': 1.88, 'away': 0.2}
Shots: {'home': 20, 'away': 4}
Shots on target: {'home': 6, 'away': 1}
Passes: {'home': 616, 'away': 347}
Corners: {'home': 8, 'away': 2}


In [13]:
# Creating a function to retrieve team lineups
def get_match_lineups(match_id):
    url = f"{BASE_URL}/matches/{match_id}/lineups"
    return get_api_data(url)

In [14]:
# Testing lineups function
match_lineups = get_match_lineups(match_id)
print(match_lineups.keys())

dict_keys(['data'])


In [15]:
# Cleaning our lineup dictionary
lineups = match_lineups["data"]

print("Home team:", lineups["home"]["name"])
print("Home formation:", lineups["home"]["formation"])
print("Away team:", lineups["away"]["name"])
print("Away formation:", lineups["away"]["formation"])

Home team: Arsenal
Home formation: 4-3-3
Away team: Coventry City
Away formation: 4-3-3


In [16]:
# Used to show not just the team formation, but the specific players in the lineup.
for player in lineups["home"]["starting_xi"]:
    print(
        player["jersey_number"],
        "-",
        player["name"],
        "-",
        player["position"]
    )

1 - David Raya - G
3 - Cristhian Mosquera - D
4 - Ben White - D
6 - Gabriel Magalhães - D
33 - Riccardo Calafiori - D
8 - Martin Ødegaard - M
41 - Declan Rice - M
49 - Myles Lewis-Skelly - M
7 - Bukayo Saka - F
17 - Christos Tzolis - F
29 - Kai Havertz - F


In [17]:
# Creating a function to retrieve specific player's match stats.
def get_match_player_stats(match_id):
    url = f"{BASE_URL}/matches/{match_id}/player-stats"
    return get_api_data(url)  

In [18]:
# Testing match player stats function.
match_player_stats = get_match_player_stats(match_id)
print(match_player_stats.keys())

dict_keys(['data'])


In [19]:
# Retrieving a team's players and looking at their match stats.
player_stats = match_player_stats["data"]

for player in player_stats:
    if player["team_id"] == "tm_9145" and player["started"]:
        print(
            player["player_name"],
            "- Rating:", player["rating"],
            "- Minutes:", player["minutes_played"]
        )

Riccardo Calafiori - Rating: 7.41 - Minutes: 81
Gabriel Magalhães - Rating: 6.09 - Minutes: 90
Ben White - Rating: 8.19 - Minutes: 90
Cristhian Mosquera - Rating: 7.1 - Minutes: 90
Myles Lewis-Skelly - Rating: 7.2 - Minutes: 90
Martin Ødegaard - Rating: 8.76 - Minutes: 76
Bukayo Saka - Rating: 7.69 - Minutes: 68
Kai Havertz - Rating: 7.93 - Minutes: 90
Christos Tzolis - Rating: 7.2 - Minutes: 76
David Raya - Rating: 7.36 - Minutes: 90
Declan Rice - Rating: 7.67 - Minutes: 68


In [20]:
# Creating a function to retrieve shotmaps
def get_match_shotmap(match_id):
    url = f"{BASE_URL}/matches/{match_id}/shotmap"
    return get_api_data(url)

In [21]:
match_shotmap = get_match_shotmap(match_id)
print(match_shotmap.keys())

dict_keys(['match_id', 'event', 'data', 'np_xg_summary', 'meta'])


In [22]:
# Cleaning up our shotmap dictionary.
shots = match_shotmap["data"]

for shot in shots[:5]:
    print(
        shot["minute"],
        "-",
        shot["player_name"],
        "-",
        shot["team_name"],
        "- x:", shot["x"],
        "- y:", shot["y"],
        "- Result:", shot["result"],
        "- xG:", shot["expected_goals"]
    )

5 - Christos Tzolis - Arsenal - x: 15.8 - y: 42 - Result: miss - xG: 0.09
7 - Bukayo Saka - Arsenal - x: 9.2 - y: 58.2 - Result: block - xG: 0.1
8 - Christos Tzolis - Arsenal - x: 15.5 - y: 49.5 - Result: save - xG: 0.08
9 - Declan Rice - Arsenal - x: 28.2 - y: 33.9 - Result: miss - xG: 0.02
13 - Loum Tchaouna - Coventry City - x: 28.4 - y: 64.9 - Result: block - xG: 0.01


In [23]:
# Creating event timeline function.
def get_match_events(match_id):
    url = f"{BASE_URL}/matches/{match_id}/timeline"
    return get_api_data(url)

In [24]:
# Testing our event timeline function.
match_events = get_match_events(match_id)
print(match_events.keys())

dict_keys(['data', 'meta'])


In [25]:
# Testing to see how data is structured, we don't make to make any assumptions.
events_data = match_events["data"]
print(type(events_data))
print(events_data.keys())

<class 'dict'>
dict_keys(['match_id', 'coverage', 'events'])


In [26]:
# Creating a shorter and more readable match timeline.
events = match_events["data"]["events"]

for event in events:
    if event["type"] in ["goal", "yellow_card", "red_card", "substitution"]:
        print(
            event["minute"],
            " -",
            event["type"],
            "-",
            event["team"]["name"] if event["team"] else None,
            "-",
            event["player"]["name"] if event["player"] else None
        )

15  - goal - Arsenal - Kai Havertz
23  - goal - Arsenal - Bukayo Saka
27  - yellow_card - Coventry City - Caleb Yirenkyi
34  - yellow_card - Arsenal - Gabriel Magalhães
49  - goal - Arsenal - Martin Ødegaard
62  - substitution - Coventry City - Victor Torp
68  - substitution - Arsenal - Noni Madueke
68  - substitution - Arsenal - Martín Zubimendi
70  - substitution - Coventry City - Jack Rudoni
70  - substitution - Coventry City - Taiwo Awoniyi
76  - substitution - Arsenal - Mikel Merino
76  - substitution - Arsenal - Eberechi Eze
81  - substitution - Arsenal - Piero Hincapié
82  - substitution - Coventry City - Gustavo Hamer


In [27]:
# Creating a function to retrieve match odds
def get_match_odds(match_id):
    url = f"{BASE_URL}/matches/{match_id}/odds"
    return get_api_data(url)

In [28]:
# Testing our odds function
match_odds = get_match_odds(match_id)
print(match_odds.keys())

dict_keys(['data'])


In [29]:
# Now that we have seen that all our URL endpoints are working, we combine all of our helper functions into one single MASTER function that can retrieve all our info with one request.

def get_full_match_data(match_id):
    return {
        "details": get_match_details(match_id),
        "statistics": get_match_statistics(match_id),
        "lineups": get_match_lineups(match_id),
        "player_stats": get_match_player_stats(match_id),
        "shotmap": get_match_shotmap(match_id),
        "events": get_match_events(match_id),
        "odds": get_match_odds(match_id)
    }

In [30]:
# Testing out our MASTER function
full_match_data = get_full_match_data(match_id)
print(full_match_data.keys())

dict_keys(['details', 'statistics', 'lineups', 'player_stats', 'shotmap', 'events', 'odds'])


In [31]:
get_full_match_data("mt_511976372")

Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5


{'details': {'data': {'id': 'mt_511976372',
   'competition_id': 'comp_3039',
   'competition_name': 'Premier League',
   'season_id': 'sn_8406098',
   'matchday': 1,
   'stage_name': None,
   'group_label': None,
   'status': 'finished',
   'utc_date': '2026-08-21T19:00:00.000Z',
   'home_team': {'id': 'tm_9145', 'name': 'Arsenal'},
   'away_team': {'id': 'tm_2954', 'name': 'Coventry City'},
   'score': {'home': 3,
    'away': 0,
    'final_score': None,
    'regulation': {'home': 3, 'away': 0},
    'after_extra_time': None,
    'penalty_shootout': None,
    'went_to_extra_time': False,
    'went_to_penalties': False,
    'winner': 'home',
    'half_time_home': 2,
    'half_time_away': 0},
   'venue': {'name': 'Emirates Stadium', 'city': 'London'},
   'is_neutral': False,
   'referee': {'id': 'ref_72946204', 'name': 'Thomas Bramall'},
   'odds_available': True,
   'live_odds_available': False,
   'xg_available': True}},
 'statistics': {'data': {'match_id': 'mt_511976372',
   'overview

In [32]:
# Automating pulling match IDs
# No more hard coding specific matches.

def get_season_matches(competition_id, season_id):
    url = (
        f"{BASE_URL}/matches"
        f"?competition_id={competition_id}"
        f"&season_id={season_id}"
    )

    return get_api_data(url)

In [33]:
# Testing getting season matches function.
season_matches = get_season_matches(
    "comp_3039",
    "sn_8406098"
)

print(season_matches.keys())

dict_keys(['data', 'meta'])


In [34]:
# Looking to see how many matches were returned from the URL
print("Matches returned:", len(season_matches["data"]))
print("Meta:", season_matches["meta"])

Matches returned: 20
Meta: {'page': 1, 'per_page': 20, 'total': 50, 'total_pages': 3}


In [35]:
for match in season_matches["data"][:10]:
    print(
        match["id"],
        "-",
        match["matchday"],
        "-",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"],
        "-",
        match["status"]
    )

mt_988360482 - 5 - Fulham vs Manchester United - scheduled
mt_899742694 - 5 - Bournemouth vs Liverpool - scheduled
mt_733622110 - 5 - Nottingham Forest vs Coventry City - scheduled
mt_899744220 - 5 - Newcastle United vs Hull City - scheduled
mt_377400558 - 5 - Manchester City vs Sunderland - scheduled
mt_644011884 - 5 - Leeds United vs Crystal Palace - scheduled
mt_200193806 - 5 - Everton vs Ipswich Town - scheduled
mt_155834755 - 5 - Brighton & Hove Albion vs Arsenal - scheduled
mt_155833448 - 5 - Tottenham Hotspur vs Aston Villa - scheduled
mt_733621034 - 5 - Brentford vs Chelsea - scheduled


In [36]:
# Building pagination, each page fits 20 matches, we currently have a total of 50, so we need 3 pages
def get_all_season_matches(competition_id, season_id):
    all_matches = []

    # We start at page 1 and adds the matches into our array: all_matches
    page = 1

    while True:
        url = (
            f"{BASE_URL}/matches"
            f"?competition_id={competition_id}"
            f"&season_id={season_id}"
            f"&page={page}"
        )

        response = get_api_data(url)

        if response is None:
            break

        # extend() function takes all the matches from that page and adds them to our master list
        all_matches.extend(response["data"])

        # API told us there are 3 pages, so Python will separate into 20 matches in the first 2 pages, then 10 in the last page and then combine them into one list of 50 matches.
        if page >= response["meta"]["total_pages"]:
            break

        page += 1

        # Gives the API a pause so we don't hit a rate limit.
        time.sleep(1)

    return all_matches

In [37]:
# Testing our match getting function
all_season_matches = get_all_season_matches(
    "comp_3039",
    "sn_8406098"
)

print("Total matches collected:", len(all_season_matches))

Total matches collected: 50


In [38]:
# Our function is showing us finished AND future scheduled matches
# We want to separate matches that have already finished and the ones that haven't been played yet.

finished_matches = [
    match for match in all_season_matches
    if match["status"] == "finished"
]

scheduled_matches = [
    match for match in all_season_matches
    if match["status"] == "scheduled"
]

print("Finished matches:", len(finished_matches))
print("Scheduled matches:", len(scheduled_matches))

Finished matches: 15
Scheduled matches: 35


In [39]:
# Let's take a look at the finished matches.

for match in finished_matches:
    print(
        match["matchday"],
        "-",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"],
        "-",
        match["score"]["home"],
        "-",
        match["score"]["away"]
    )

2 - Tottenham Hotspur vs Newcastle United - 0 - 2
2 - Coventry City vs Hull City - 0 - 1
2 - Bournemouth vs Everton - 1 - 1
2 - Liverpool vs Nottingham Forest - 2 - 2
2 - Crystal Palace vs Manchester City - 1 - 4
1 - Fulham vs Chelsea - 2 - 3
1 - Newcastle United vs Liverpool - 2 - 2
1 - Manchester City vs Bournemouth - 2 - 1
1 - Brighton & Hove Albion vs Aston Villa - 4 - 0
1 - Brentford vs Tottenham Hotspur - 3 - 0
1 - Ipswich Town vs Sunderland - 2 - 1
1 - Everton vs Crystal Palace - 2 - 0
1 - Nottingham Forest vs Leeds United - 0 - 1
1 - Hull City vs Manchester United - 2 - 0
1 - Arsenal vs Coventry City - 3 - 0


In [40]:
# Create a lighter function to inspect past matches (historical modeling)

def get_model_match_data(match_id):
    full_data = {}

    full_data["details"] = get_match_details(match_id)
    time.sleep(2)

    full_data["statistics"] = get_match_statistics(match_id)
    time.sleep(2)

    full_data["lineups"] = get_match_lineups(match_id)
    time.sleep(2)

    full_data["player_stats"] = get_match_player_stats(match_id)
    time.sleep(2)

    full_data["odds"] = get_match_odds(match_id)

    missing = [
        section
        for section, value in full_data.items()
        if value is None
    ]

    if missing:
        print("Incomplete match data. Missing:", missing)
        return None

    return full_data

In [41]:
# Testing our function on one completed match.
test_model_data = get_model_match_data(finished_matches[0]["id"])
print(test_model_data.keys())

dict_keys(['details', 'statistics', 'lineups', 'player_stats', 'odds'])


In [42]:
# Saving our matches

import json
from pathlib import Path

In [43]:
RAW_MATCH_DIR = Path("../data/raw/matches")
RAW_MATCH_DIR.mkdir(parents = True, exist_ok = True)
print(RAW_MATCH_DIR)

..\data\raw\matches


In [44]:
# Creating a function that saves the match into the correct folder.

def save_match_data(match_id, match_data, save_dir):
    # Creating the filename.
    file_path = save_dir / f"{match_id}.json"

    # We open the file and write (w)
    # utf-8 makes sure special characters are also stored.
    with open(file_path, "w", encoding = "utf-8") as f:
        # match_data is a Python dictionary, so json.dump takes it and writes it into a .json file.
        json.dump(match_data, f, indent = 2, ensure_ascii = False)

    return file_path

In [45]:
# Save our first match

test_match_id = finished_matches[0]["id"]

saved_path = save_match_data(
    test_match_id,
    test_model_data,
    RAW_MATCH_DIR / "2026_27"
)

print("Saved to:", saved_path)

Saved to: ..\data\raw\matches\2026_27\mt_644018139.json


In [46]:
# Automating the entire saving matches process.

# Going through every finished match in our list.
for match in finished_matches:
    match_id = match["id"]
    file_path = RAW_MATCH_DIR / "2026_27" / f"{match_id}.json"

    # If the match has already been saved, we skip that match.
    if file_path.exists():
        print("Skipping:", match_id, "- already saved")
        continue

    # If the match doesn't already exist, we save the match.
    print(
        "Collecting:",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"]
    )

    # Pulls the model data
    match_data = get_model_match_data(match_id)

    # Saves the data
    if match_data is not None:
        save_match_data(
            match_id, 
            match_data,
            RAW_MATCH_DIR / "2026_27"
        )
        
        print("Saved:", match_id)

    # 2 second pause to avoid hitting the rate limit.
    time.sleep(2)

Skipping: mt_644018139 - already saved
Collecting: Coventry City vs Hull City
Saved: mt_155834351
Collecting: Bournemouth vs Everton
Saved: mt_377405071
Collecting: Liverpool vs Nottingham Forest
Saved: mt_022587842
Collecting: Crystal Palace vs Manchester City
Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5
Rate limit reached. Waiting 20 seconds before retry 3/5
Saved: mt_988360616
Skipping: mt_022587821 - already saved
Skipping: mt_644018143 - already saved
Skipping: mt_511976717 - already saved
Skipping: mt_644018124 - already saved
Skipping: mt_988360680 - already saved
Skipping: mt_644018510 - already saved
Skipping: mt_466259151 - already saved
Skipping: mt_899742430 - already saved
Skipping: mt_022587983 - already saved
Skipping: mt_511976372 - already saved


In [47]:
# Let's check if everything was saved properly

for file_path in RAW_MATCH_DIR.glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        data = json.load(f)

    missing = [
        section
        for section, value in data.items()
        if value is None
    ]

    if missing:
        print(file_path.name, "- MISSING:", missing)
    else:
        print(file_path.name, "- COMPLETE")

In [48]:
import pandas as pd

In [49]:
# A JSON file might be too nested for machine learning purposes.
# This function takes a JSON dictionary and turns it into something flat where each match becomes a row in our future dataset.
def extract_match_row(match_data):
    details = match_data["details"]["data"]
    stats = match_data["statistics"]["data"]
    lineups = match_data["lineups"]["data"]

    return {
        "match_id": details["id"],
        "date": details["utc_date"],
        "matchday": details["matchday"],

        "home_team": details["home_team"]["name"],
        "away_team": details["away_team"]["name"],

        "home_goals": details["score"]["home"],
        "away_goals": details["score"]["away"],

        "home_xg": stats["overview"]["expected_goals"]["all"]["home"],
        "away_xg": stats["overview"]["expected_goals"]["all"]["away"],

        "home_shots": stats["overview"]["total_shots"]["all"]["home"],
        "away_shots": stats["overview"]["total_shots"]["all"]["away"],

        "home_shots_on_target": stats["overview"]["shots_on_target"]["all"]["home"],
        "away_shots_on_target": stats["overview"]["shots_on_target"]["all"]["away"],

        "home_possession": stats["overview"]["ball_possession"]["all"]["home"],
        "away_possession": stats["overview"]["ball_possession"]["all"]["away"],

        "home_passes": stats["overview"]["passes"]["all"]["home"],
        "away_passes": stats["overview"]["passes"]["all"]["away"],

        "home_corners": stats["overview"]["corner_kicks"]["all"]["home"],
        "away_corners": stats["overview"]["corner_kicks"]["all"]["away"],

        "home_formation": lineups["home"]["formation"],
        "away_formation": lineups["away"]["formation"],

        "winner": details["score"]["winner"]
    }

In [50]:
# Now each row will correspond to a JSON file

match_rows = []

for file_path in (RAW_MATCH_DIR / "2026_27").glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        match_data = json.load(f)

    row = extract_match_row(match_data)
    match_rows.append(row)


matches_df = pd.DataFrame(match_rows)

In [51]:
# Inspecting our match-level DataFrame
print(matches_df.shape)
matches_df.head()

(15, 22)


,match_id,date,matchday,home_team,away_team,home_goals,away_goals,home_xg,away_xg,home_shots,...,away_shots_on_target,home_possession,away_possession,home_passes,away_passes,home_corners,away_corners,home_formation,away_formation,winner
0,mt_022587821,2026-08-24T19:00:00.000Z,1,Fulham,Chelsea,2,3,1.33,2.24,14,...,6,62,38,639,394,6,4,4-2-3-1,3-4-3,away
1,mt_022587842,2026-08-29T11:30:00.000Z,2,Liverpool,Nottingham Forest,2,2,1.61,2.30,13,...,3,70,30,625,281,3,5,4-2-3-1,3-4-2-1,draw
2,mt_022587983,2026-08-22T11:30:00.000Z,1,Hull City,Manchester United,2,0,1.26,1.81,8,...,5,29,71,253,654,1,6,3-4-2-1,4-2-3-1,home
3,mt_155834351,2026-08-29T14:00:00.000Z,2,Coventry City,Hull City,0,1,1.28,0.73,11,...,2,69,31,570,262,6,2,4-2-3-1,5-4-1,away
4,mt_377405071,2026-08-29T14:00:00.000Z,2,Bournemouth,Everton,1,1,2.14,1.72,17,...,7,59,41,444,312,10,7,4-2-3-1,4-2-3-1,draw


In [52]:
# Check to see if all our variables have the correct data type.
matches_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   match_id              15 non-null     str    
 1   date                  15 non-null     str    
 2   matchday              15 non-null     int64  
 3   home_team             15 non-null     str    
 4   away_team             15 non-null     str    
 5   home_goals            15 non-null     int64  
 6   away_goals            15 non-null     int64  
 7   home_xg               15 non-null     float64
 8   away_xg               15 non-null     float64
 9   home_shots            15 non-null     int64  
 10  away_shots            15 non-null     int64  
 11  home_shots_on_target  15 non-null     int64  
 12  away_shots_on_target  15 non-null     int64  
 13  home_possession       15 non-null     int64  
 14  away_possession       15 non-null     int64  
 15  home_passes           15 non-null   

In [53]:
# We see if there are any null valuables in our columns.
matches_df.isnull().sum()

match_id                0
date                    0
matchday                0
home_team               0
away_team               0
home_goals              0
away_goals              0
home_xg                 0
away_xg                 0
home_shots              0
away_shots              0
home_shots_on_target    0
away_shots_on_target    0
home_possession         0
away_possession         0
home_passes             0
away_passes             0
home_corners            0
away_corners            0
home_formation          0
away_formation          0
winner                  0
dtype: int64

In [54]:
matches_df["date"] = pd.to_datetime(matches_df["date"], utc = True)
matches_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   match_id              15 non-null     str                
 1   date                  15 non-null     datetime64[us, UTC]
 2   matchday              15 non-null     int64              
 3   home_team             15 non-null     str                
 4   away_team             15 non-null     str                
 5   home_goals            15 non-null     int64              
 6   away_goals            15 non-null     int64              
 7   home_xg               15 non-null     float64            
 8   away_xg               15 non-null     float64            
 9   home_shots            15 non-null     int64              
 10  away_shots            15 non-null     int64              
 11  home_shots_on_target  15 non-null     int64              
 12  away_shots_on_target 

In [55]:
# Moving from raw data to processed data.
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

In [56]:
processed_file = PROCESSED_DIR / "matches.csv"

matches_df.to_csv(
    processed_file,

    # Prevents pandas from adding unnecessary columns containing 0,1,2,3,..
    index = False
)

print("Saved to:", processed_file)

Saved to: ..\data\processed\matches.csv


In [57]:
# Moving onto historical data (Past seasons)
PREMIER_LEAGUE_ID = "comp_3039"

SEASONS = {
    "2026_27": "sn_8406098",
    "2025_26": "sn_6125938",
    "2024_25": "sn_3057848",
    "2023_24": "sn_606923",
    "2022_23": "sn_654318",
    "2021_22": "sn_699021",
    "2020_21": "sn_477656"
}

In [78]:
collect_season_baseline_data(
    "2022_23",
    finished_2022_23
)

[1/380] Collecting: Leicester City vs West Ham United
Saved: mt_587295251
[2/380] Collecting: Southampton vs Liverpool
Saved: mt_259612629
[3/380] Collecting: Crystal Palace vs Nottingham Forest
Saved: mt_360843842
[4/380] Collecting: Leeds United vs Tottenham Hotspur
Saved: mt_621306309
[5/380] Collecting: Everton vs Bournemouth
Saved: mt_018450452
[6/380] Collecting: Manchester United vs Fulham
Saved: mt_360843821
[7/380] Collecting: Arsenal vs Wolverhampton
Saved: mt_976139140
[8/380] Collecting: Brentford vs Manchester City
Saved: mt_259612672
[9/380] Collecting: Aston Villa vs Brighton & Hove Albion
Saved: mt_193081068
[10/380] Collecting: Chelsea vs Newcastle United
Saved: mt_742967950
[11/380] Collecting: Manchester United vs Chelsea
Saved: mt_250997231
[12/380] Collecting: Brighton & Hove Albion vs Manchester City
Saved: mt_978662909
[13/380] Collecting: Newcastle United vs Leicester City
Saved: mt_012881773
[14/380] Collecting: Manchester City vs Chelsea
Rate limit reached. Wa

In [64]:
matches_2022_23 = season_2022_23_matches["data"]

finished_2022_23 = [
    match for match in matches_2022_23
    if match["status"] == "finished"
]

print("Finished matches:", len(finished_2022_23))

test_match = finished_2022_23[0]

print("Test match ID:", test_match["id"])
print(
    test_match["home_team"]["name"],
    "vs",
    test_match["away_team"]["name"]
)

Finished matches: 20
Test match ID: mt_587295251
Leicester City vs West Ham United


In [91]:
# Testing on a past season to see if all matches are returned to us from the API.
# For a completed season, the total amount of games is 380.
matches_2025_26 = get_all_season_matches(
    PREMIER_LEAGUE_ID,
    SEASONS["2025_26"]
)

print("Matches found:", len(matches_2025_26))

Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5
Rate limit reached. Waiting 20 seconds before retry 3/5
Matches found: 380


In [92]:
from collections import Counter

status_counts = Counter(
    match["status"] for match in matches_2025_26
)

print(status_counts)

Counter({'finished': 380})


In [ ]:
# Used as a sanity check to make sure we have pulled matches from the correct season (2025-26)
for match in matches_2025_26[:5]:
    print(
        match["matchday"],
        "-",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"],
        "-",
        match["status"]
    )

38 - West Ham United vs Leeds United - finished
38 - Tottenham Hotspur vs Everton - finished
38 - Sunderland vs Chelsea - finished
38 - Fulham vs Newcastle United - finished
38 - Nottingham Forest vs Bournemouth - finished


In [86]:
# Creating a historical collector.
def get_baseline_match_data(match_id):
    match_data = {}

    match_data["details"] = get_match_details(match_id)
    time.sleep(2)

    match_data["statistics"] = get_match_statistics(match_id)
    time.sleep(2)

    match_data["odds"] = get_match_odds(match_id)

    # Details are required because we need the actual match/result.
    if match_data["details"] is None:
        print("Missing match details. Skipping:", match_id)
        return None

    # Statistics and odds can be missing and will be handled later as NaN.
    missing = [
        section
        for section in ["statistics", "odds"]
        if match_data[section] is None
    ]

    if missing:
        print("Saving with missing sections:", missing)

    return match_data

In [76]:
# Creating folders for each season.
SEASON_DIRS = {}

# Looping through the seasons we listed earlier.
# Then creating a path for each season
for season_name in SEASONS:
    season_dir = RAW_MATCH_DIR / season_name
    season_dir.mkdir(parents = True, exist_ok = True)
    SEASON_DIRS[season_name] = season_dir

print(SEASON_DIRS)

{'2026_27': WindowsPath('../data/raw/matches/2026_27'), '2025_26': WindowsPath('../data/raw/matches/2025_26'), '2024_25': WindowsPath('../data/raw/matches/2024_25'), '2023_24': WindowsPath('../data/raw/matches/2023_24'), '2022_23': WindowsPath('../data/raw/matches/2022_23'), '2021_22': WindowsPath('../data/raw/matches/2021_22'), '2020_21': WindowsPath('../data/raw/matches/2020_21')}


In [74]:
# Building a collector for the historical seasons.

def collect_season_baseline_data(season_name, matches):
    season_dir = SEASON_DIRS[season_name]

    total_matches = len(matches)

    # Adds a progress counter, just to keep track of how many matches are left to save.
    for i, match in enumerate(matches, start = 1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        # The collrector is resumable, so if it stops collecting today, you can continue collecting tomorrow.
        if file_path.exists():
            print(f"[{i}/{total_matches}] Skipping {match_id} - already saved")
            continue

        print(
            f"[{i}/{total_matches}] Collecting:",
            match["home_team"]["name"],
            "vs",
            match["away_team"]["name"]
        )

        match_data = get_baseline_match_data(match_id)

        if match_data is not None:
            with open(file_path, "w", encoding = "utf-8") as f:
                json.dump(match_data, f, indent = 2, ensure_ascii = False)

            print("Saved:", match_id)
        else:
            print("Not saved because data was incomplete:", match_id)

        time.sleep(5)

In [ ]:
# Testing to see if the historical seasons collector works.
collect_season_baseline_data(
    "2025_26",
    matches_2025_26[:3]
)

[1/3] Collecting: West Ham United vs Leeds United
Saved: mt_838955396
[2/3] Collecting: Tottenham Hotspur vs Everton
Saved: mt_363788671
[3/3] Collecting: Sunderland vs Chelsea
Saved: mt_626433243


In [101]:
# Testing the collector on 3 matches worked, so we can get rid of the [:3] limit.
collect_season_baseline_data(
    "2025_26",
    matches_2025_26
)

1/380] Skipping mt_838955396 - already saved
2/380] Skipping mt_363788671 - already saved
3/380] Skipping mt_626433243 - already saved
4/380] Skipping mt_404677066 - already saved
5/380] Skipping mt_010244121 - already saved
6/380] Skipping mt_585122817 - already saved
7/380] Skipping mt_979811780 - already saved
8/380] Skipping mt_252066552 - already saved
9/380] Skipping mt_191500998 - already saved
10/380] Skipping mt_747399440 - already saved
11/380] Skipping mt_377685036 - already saved
12/380] Skipping mt_626433281 - already saved
13/380] Skipping mt_191500937 - already saved
14/380] Skipping mt_363788668 - already saved
15/380] Skipping mt_838955330 - already saved
16/380] Skipping mt_404677000 - already saved
17/380] Skipping mt_010244118 - already saved
18/380] Skipping mt_585122886 - already saved
19/380] Skipping mt_626433224 - already saved
20/380] Skipping mt_838955483 - already saved
21/380] Skipping mt_899285792 - already saved
22/380] Skipping mt_010244187 - already sav

In [102]:
# Verify the 2025/26 season folder has 380 JSON files.
season_2025_dir = SEASON_DIRS["2025_26"]
saved_files = list(season_2025_dir.glob("*.json"))

print("Saved matches:", len(saved_files))

Saved matches: 380


In [103]:
# Perform an integrity check
# Making sure every JSON file contains the sections we want to look at

incomplete_files = []

for file_path in SEASON_DIRS["2025_26"].glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        data = json.load(f)

    missing = [
        section
        for section in ["details", "statistics", "odds"]
        if data.get(section) is None
    ]

    if missing:
        incomplete_files.append((file_path.name, missing))

print("Total files:", 380)
print("Incomplete files:", len(incomplete_files))

if incomplete_files:
    for file_name, missing in incomplete_files:
        print(file_name, "- MISSING:", missing)
else:
    print("All 2025/26 files are complete.")

Total files: 380
Incomplete files: 0
All 2025/26 files are complete.


In [104]:
# Now collecting historical data from the 2024/25 season
matches_2024_25 = get_all_season_matches(
    PREMIER_LEAGUE_ID,
    SEASONS["2024_25"]
)

print("Matches found:", len(matches_2024_25))

Matches found: 381


In [105]:
# Our last cell returned 381 matches, so we have to check for any duplicates

status_counts_2024_25 = Counter(
    match["status"] for match in matches_2024_25
)

print(status_counts_2024_25)

Counter({'finished': 380, 'postponed': 1})


In [106]:
match_ids_2024_25 = [
    match["id"]
    for match in matches_2024_25
]

print("Total matches:", len(match_ids_2024_25))
print("Unique match IDs:", len(set(match_ids_2024_25)))

Total matches: 381
Unique match IDs: 381


In [107]:
for match in matches_2024_25:
    if match["status"] != "finished":
        print(
            match["id"],
            match["matchday"],
            match["home_team"]["name"],
            "vs",
            match["away_team"]["name"],
            "-",
            match["status"]
        )

mt_408687117 37 Manchester City vs Bournemouth - postponed


In [109]:
# We found the duplicate record, let's see if the match ID shows that it was finished
for match in matches_2024_25:
    if (
        match["home_team"]["name"] == "Manchester City"
        and match["away_team"]["name"] == "Bournemouth"
    ):
        print(
            match["id"],
            "- Matchday:", match["matchday"],
            "- Status:", match["status"]
        )

mt_838200836 - Matchday: 37 - Status: finished
mt_408687117 - Matchday: 37 - Status: postponed


In [110]:
# Creating a clean list
# Removing the postoned match ID

matches_2024_25_clean = [
    match
    for match in matches_2024_25
    if match["status"] == "finished"
]

print("Original matches:", len(matches_2024_25))
print("Finished matches:", len(matches_2024_25_clean))

Original matches: 381
Finished matches: 380


In [111]:
# Run the collector function for the 2024/25 season.

collect_season_baseline_data(
    "2024_25",
    matches_2024_25_clean
)

[1/380] Collecting: Fulham vs Manchester City
Saved: mt_745359007
[2/380] Collecting: Tottenham Hotspur vs Brighton & Hove Albion
Saved: mt_361718222
[3/380] Collecting: Nottingham Forest vs Chelsea
Saved: mt_629493559
[4/380] Collecting: Southampton vs Arsenal
Saved: mt_745359056
[5/380] Collecting: Liverpool vs Crystal Palace
Saved: mt_830905606
[6/380] Collecting: Wolverhampton vs Brentford
Saved: mt_972821420
[7/380] Collecting: Newcastle United vs Everton
Saved: mt_257076862
[8/380] Collecting: Manchester United vs Aston Villa
Saved: mt_196560708
[9/380] Collecting: Ipswich Town vs West Ham United
Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5
Saved: mt_830905650
[10/380] Collecting: Bournemouth vs Leicester City
Saved: mt_629493534
[11/380] Collecting: Manchester City vs Bournemouth
Saved: mt_838200836
[12/380] Collecting: Crystal Palace vs Wolverhampton
Saved: mt_972821362
[13/380] Collecting: Brighton & Hove Albion

In [112]:
# Verify that all matches in season 2024/25 saved properly.

season_2024_dir = SEASON_DIRS["2024_25"]

saved_files = list(season_2024_dir.glob("*.json"))

print("Saved matches:", len(saved_files))

Saved matches: 380


In [113]:
# Check for incomplete files
incomplete_files = []

for file_path in SEASON_DIRS["2024_25"].glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        data = json.load(f)

    missing = [
        section
        for section in ["details", "statistics", "odds"]
        if data.get(section) is None
    ]

    if missing:
        incomplete_files.append((file_path.name, missing))

print("Total files:", len(saved_files))
print("Incomplete files:", len(incomplete_files))

if incomplete_files:
    for file_name, missing in incomplete_files:
        print(file_name, "- MISSING:", missing)
else:
    print("All 2024/25 files are complete.")

Total files: 380
Incomplete files: 0
All 2024/25 files are complete.


In [114]:
# Start collecting historical data from season 2023/24
# First conduct an initial check on that season.
matches_2023_24 = get_all_season_matches(
    PREMIER_LEAGUE_ID,
    SEASONS["2023_24"]
)

print("Matches found:", len(matches_2023_24))

Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5
Matches found: 380


In [115]:
status_counts_2023_24 = Counter(
    match["status"] for match in matches_2023_24
)

print(status_counts_2023_24)

Counter({'finished': 380})


In [116]:
# This season doesn't need any filtering
# Go ahead and collect all the data from that season.

collect_season_baseline_data(
    "2023_24",
    matches_2023_24
)

[1/380] Collecting: Sheffield United vs Tottenham Hotspur
Saved: mt_012211518
[2/380] Collecting: Manchester City vs West Ham United
Saved: mt_250055191
[3/380] Collecting: Luton Town vs Fulham
Saved: mt_839933745
[4/380] Collecting: Liverpool vs Wolverhampton
Saved: mt_406600255
[5/380] Collecting: Crystal Palace vs Aston Villa
Saved: mt_978877303
[6/380] Collecting: Chelsea vs Bournemouth
Saved: mt_839933729
[7/380] Collecting: Burnley vs Nottingham Forest
Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5
Saved: mt_406600299
[8/380] Collecting: Brighton & Hove Albion vs Manchester United
Saved: mt_978877494
[9/380] Collecting: Brentford vs Newcastle United
Saved: mt_743344073
[10/380] Collecting: Arsenal vs Everton
Saved: mt_624422568
[11/380] Collecting: Manchester United vs Newcastle United
Rate limit reached. Waiting 5 seconds before retry 1/5
Saved: mt_257357296
[12/380] Collecting: Brighton & Hove Albion vs Chelsea
Sav

In [313]:
# ---------------------------------------------------------
# GET 2026/27 PREMIER LEAGUE MATCH LIST
# ---------------------------------------------------------

matches_2026_27 = get_all_season_matches(
    "comp_3039",
    "sn_8406098"
)

print(
    "2026/27 Premier League matches found:",
    len(matches_2026_27)
)

2026/27 Premier League matches found: 60


In [314]:
print("Total returned:", len(matches_2026_27))
print()

for match in matches_2026_27[:10]:
    print(
        match.get("id"),
        match.get("date"),
        match.get("status"),
        match.get("home_team", {}).get("name"),
        "vs",
        match.get("away_team", {}).get("name")
    )

Total returned: 60

mt_022588781 None scheduled Coventry City vs Newcastle United
mt_899744246 None scheduled Liverpool vs Manchester City
mt_644011813 None scheduled Hull City vs Everton
mt_466255956 None scheduled Crystal Palace vs Nottingham Forest
mt_733622126 None scheduled Manchester United vs Tottenham Hotspur
mt_155833431 None scheduled Sunderland vs Brighton & Hove Albion
mt_377400501 None scheduled Ipswich Town vs Fulham
mt_511977677 None scheduled Chelsea vs Bournemouth
mt_988366060 None scheduled Aston Villa vs Brentford
mt_200199332 None scheduled Arsenal vs Leeds United


In [315]:
from collections import Counter

status_counts = Counter(
    match.get("status")
    for match in matches_2026_27
)

print("STATUS COUNTS")
print()

for status, count in status_counts.items():
    print(status, count)

STATUS COUNTS

scheduled 40
finished 20


In [316]:
# ---------------------------------------------------------
# KEEP ONLY FINISHED 2026/27 PREMIER LEAGUE MATCHES
# ---------------------------------------------------------

finished_matches_2026_27 = [
    match
    for match in matches_2026_27
    if match.get("status") == "finished"
]


print(
    "Finished 2026/27 Premier League matches:",
    len(finished_matches_2026_27)
)

Finished 2026/27 Premier League matches: 20


In [317]:
# ---------------------------------------------------------
# COLLECT / RESUME FINISHED 2026/27 MATCHES
# ---------------------------------------------------------

collect_season_baseline_data(
    "2026_27",
    finished_matches_2026_27
)

[1/20] Collecting: Aston Villa vs Arsenal
Saved: mt_466259566
[2/20] Collecting: Manchester United vs Ipswich Town
Saved: mt_511976728
[3/20] Collecting: Leeds United vs Brentford
Saved: mt_644018195
[4/20] Collecting: Sunderland vs Fulham
Saved: mt_466259577
[5/20] Collecting: Chelsea vs Brighton & Hove Albion
Saved: mt_733621236
[6/20] Skipping mt_644018139 - already saved
[7/20] Skipping mt_155834351 - already saved
[8/20] Skipping mt_377405071 - already saved
[9/20] Skipping mt_022587842 - already saved
[10/20] Skipping mt_988360616 - already saved
[11/20] Skipping mt_022587821 - already saved
[12/20] Skipping mt_644018143 - already saved
[13/20] Skipping mt_511976717 - already saved
[14/20] Skipping mt_644018124 - already saved
[15/20] Skipping mt_988360680 - already saved
[16/20] Skipping mt_644018510 - already saved
[17/20] Skipping mt_466259151 - already saved
[18/20] Skipping mt_899742430 - already saved
[19/20] Skipping mt_022587983 - already saved
[20/20] Skipping mt_5119763

In [321]:
import json
import pandas as pd
from pathlib import Path


# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------

RAW_MATCH_DIR = Path("../data/raw/matches")

PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# SAFE STAT EXTRACTOR
# ---------------------------------------------------------

def get_stat(statistics, team_side, stat_name):

    if statistics is None:
        return None

    if isinstance(statistics, dict):
        statistics = statistics.get(
            "data",
            statistics
        )

    if not isinstance(statistics, dict):
        return None

    team_stats = statistics.get(
        team_side,
        {}
    )

    if not isinstance(team_stats, dict):
        return None

    value = team_stats.get(
        stat_name
    )

    return value


# ---------------------------------------------------------
# PROCESS ONE RAW MATCH FILE
# ---------------------------------------------------------

def process_match_file(file_path):

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            match_data = json.load(f)

    except Exception as e:

        print(
            f"Could not read {file_path}: {e}"
        )

        return None


    # -----------------------------------------------------
    # DETAILS
    # -----------------------------------------------------

    details = match_data.get(
        "details"
    )

    if details is None:
        return None

    if isinstance(details, dict):

        details = details.get(
            "data",
            details
        )

    if not isinstance(details, dict):
        return None


    # Only completed matches
    if details.get("status") != "finished":
        return None


    # -----------------------------------------------------
    # STATISTICS
    # -----------------------------------------------------

    statistics = match_data.get(
        "statistics"
    )


    # -----------------------------------------------------
    # TEAM INFORMATION
    # -----------------------------------------------------

    home_team_data = details.get(
        "home_team",
        {}
    )

    away_team_data = details.get(
        "away_team",
        {}
    )


    home_team = (
        home_team_data.get("name")
        if isinstance(
            home_team_data,
            dict
        )
        else None
    )

    away_team = (
        away_team_data.get("name")
        if isinstance(
            away_team_data,
            dict
        )
        else None
    )


    # -----------------------------------------------------
    # SCORE
    # -----------------------------------------------------

    home_goals = details.get(
        "home_score"
    )

    away_goals = details.get(
        "away_score"
    )


    # Some API responses may store score differently
    if isinstance(home_goals, dict):

        home_goals = (
            home_goals.get("current")
            or home_goals.get("display")
        )


    if isinstance(away_goals, dict):

        away_goals = (
            away_goals.get("current")
            or away_goals.get("display")
        )


    # -----------------------------------------------------
    # WINNER
    # -----------------------------------------------------

    winner = None

    if (
        home_goals is not None
        and away_goals is not None
    ):

        try:

            if home_goals > away_goals:
                winner = "H"

            elif away_goals > home_goals:
                winner = "A"

            else:
                winner = "D"

        except TypeError:
            winner = None


    # -----------------------------------------------------
    # FOLDER INFORMATION
    #
    # .../matches/competition/season/file.json
    # -----------------------------------------------------

    season = file_path.parent.name

    competition = (
        file_path
        .parent
        .parent
        .name
    )


    # -----------------------------------------------------
    # BUILD ROW
    # -----------------------------------------------------

    row = {

        "match_id":
            details.get("id"),

        "date":
            details.get("date"),

        "season":
            season,

        "competition":
            competition,

        "matchday":
            details.get("matchday"),

        "home_team":
            home_team,

        "away_team":
            away_team,

        "home_goals":
            home_goals,

        "away_goals":
            away_goals,

        "home_xg":
            get_stat(
                statistics,
                "home",
                "expected_goals"
            ),

        "away_xg":
            get_stat(
                statistics,
                "away",
                "expected_goals"
            ),

        "home_shots":
            get_stat(
                statistics,
                "home",
                "shots"
            ),

        "away_shots":
            get_stat(
                statistics,
                "away",
                "shots"
            ),

        "home_shots_on_target":
            get_stat(
                statistics,
                "home",
                "shots_on_target"
            ),

        "away_shots_on_target":
            get_stat(
                statistics,
                "away",
                "shots_on_target"
            ),

        "home_possession":
            get_stat(
                statistics,
                "home",
                "ball_possession"
            ),

        "away_possession":
            get_stat(
                statistics,
                "away",
                "ball_possession"
            ),

        "home_passes":
            get_stat(
                statistics,
                "home",
                "passes"
            ),

        "away_passes":
            get_stat(
                statistics,
                "away",
                "passes"
            ),

        "home_corners":
            get_stat(
                statistics,
                "home",
                "corners"
            ),

        "away_corners":
            get_stat(
                statistics,
                "away",
                "corners"
            ),

        "winner":
            winner
    }


    return row


# ---------------------------------------------------------
# PROCESS ALL RAW JSON FILES
# ---------------------------------------------------------

rows = []


for file_path in RAW_MATCH_DIR.rglob("*.json"):

    row = process_match_file(
        file_path
    )

    if row is not None:
        rows.append(row)


historical_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# REMOVE DUPLICATE MATCH IDS
# ---------------------------------------------------------

historical_df = (
    historical_df
    .drop_duplicates(
        subset="match_id",
        keep="last"
    )
    .copy()
)


# ---------------------------------------------------------
# SORT
# ---------------------------------------------------------

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

output_path = (
    PROCESSED_DIR
    / "historical_matches.csv"
)


historical_df.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# CHECK RESULTS
# ---------------------------------------------------------

print(
    "Historical shape:",
    historical_df.shape
)

print()

print(
    "COMPETITION COUNTS"
)

print()

print(
    historical_df[
        "competition"
    ]
    .value_counts()
)

print()

print(
    "Premier League by season"
)

print()

print(
    historical_df[
        historical_df[
            "competition"
        ]
        == "premier_league"
    ][
        "season"
    ]
    .value_counts()
    .sort_index()
)

print()

print(
    "Saved:",
    output_path
)

Historical shape: (10673, 22)

COMPETITION COUNTS

competition
championship         3342
premier_league       2300
europa_league        1426
champions_league     1383
conference_league     869
fa_cup                798
efl_cup               555
Name: count, dtype: int64

Premier League by season

season
2020_21    380
2021_22    380
2022_23    380
2023_24    380
2024_25    380
2025_26    380
2026_27     20
Name: count, dtype: int64

Saved: ..\data\processed\historical_matches.csv


In [117]:
# Verify all matches were saved.

season_2023_dir = SEASON_DIRS["2023_24"]

saved_files = list(season_2023_dir.glob("*.json"))

print("Saved matches:", len(saved_files))

Saved matches: 380


In [118]:
incomplete_files = []

for file_path in SEASON_DIRS["2023_24"].glob("*.json"):
    with open(file_path, "r", encoding = "utf-8") as f:
        data = json.load(f)

    missing = [
        section
        for section in ["details", "statistics", "odds"]
        if data.get(section) is None
    ]

    if missing:
        incomplete_files.append((file_path.name, missing))

print("Total files:", len(saved_files))
print("incomplete files:", len(incomplete_files))

if incomplete_files:
    for file_name, missing in incomplete_files:
        print(file_name, "- MISSING:", missing)
else:
    print("All 2023/24 files are complete.")

Total files: 380
incomplete files: 0
All 2023/24 files are complete.


In [119]:
import shutil

current_season_dir = SEASON_DIRS["2026_27"]

for file_path in RAW_MATCH_DIR.glob("*.json"):
    destination = current_season_dir / file_path.name

    shutil.move(str(file_path), str(destination))

    print("Moved:", file_path.name)

Moved: mt_022587821.json
Moved: mt_022587983.json
Moved: mt_466259151.json
Moved: mt_511976372.json
Moved: mt_511976717.json
Moved: mt_644018124.json
Moved: mt_644018143.json
Moved: mt_644018510.json
Moved: mt_899742430.json
Moved: mt_988360680.json


In [189]:
def get_stat(overview, stat_name, side):
    stat = overview.get(stat_name)

    if stat is None:
        return None

    full_match = stat.get("all")

    if full_match is None:
        return None

    return full_match.get(side)

In [190]:
missing_xg_matches = []

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding="utf-8") as f:
            match_data = json.load(f)

        details = match_data["details"]["data"]
        overview = match_data["statistics"]["data"]["overview"]

        expected_goals = overview.get("expected_goals")

        if expected_goals is None or expected_goals.get("all") is None:
            missing_xg_matches.append({
                "season": season_name,
                "match_id": details["id"],
                "home_team": details["home_team"]["name"],
                "away_team": details["away_team"]["name"]
            })

print("Matches missing xG:", len(missing_xg_matches))

for match in missing_xg_matches:
    print(match)

Matches missing xG: 2
{'season': '2023_24', 'match_id': 'mt_581188321', 'home_team': 'West Ham United', 'away_team': 'Luton Town'}
{'season': '2023_24', 'match_id': 'mt_978877432', 'home_team': 'Tottenham Hotspur', 'away_team': 'Burnley'}


In [197]:
# Helper function for matches with no xG.
def get_stat(stats, stat_name, side):
    stat = stats["overview"].get(stat_name)

    if stat is None or stat.get("all") is None:
        return None

    return stat["all"][side]

In [198]:
def extract_historical_match_row(match_data):
    details = match_data["details"]["data"]
    stats = match_data["statistics"]["data"]

    return {
        "match_id": details["id"],
        "date": details["utc_date"],
        "matchday": details["matchday"],

        "home_team": details["home_team"]["name"],
        "away_team": details["away_team"]["name"],

        "home_goals": details["score"]["home"],
        "away_goals": details["score"]["away"],

        "home_xg": get_stat(stats, "expected_goals", "home"),
        "away_xg": get_stat(stats, "expected_goals", "away"),

        "home_shots": get_stat(stats, "total_shots", "home"),
        "away_shots": get_stat(stats, "total_shots", "away"),

        "home_shots_on_target": get_stat(stats, "shots_on_target", "home"),
        "away_shots_on_target": get_stat(stats, "shots_on_target", "away"),

        "home_possession": get_stat(stats, "ball_possession", "home"),
        "away_possession": get_stat(stats, "ball_possession", "away"),

        "home_passes": get_stat(stats, "passes", "home"),
        "away_passes": get_stat(stats, "passes", "away"),

        "home_corners": get_stat(stats, "corner_kicks", "home"),
        "away_corners": get_stat(stats, "corner_kicks", "away"),

        "winner": details["score"]["winner"]
    }

In [199]:
historical_rows = []

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding="utf-8") as f:
            match_data = json.load(f)

        row = extract_historical_match_row(match_data)
        row["season"] = season_name

        historical_rows.append(row)

print("Historical rows collected:", len(historical_rows))

Historical rows collected: 1140


In [200]:
# Turning our historical seasons rows into a DataFrame

historical_df = pd.DataFrame(historical_rows)

historical_df.shape

(1140, 21)

In [201]:
historical_df.head()

,match_id,date,matchday,home_team,away_team,home_goals,away_goals,home_xg,away_xg,home_shots,...,home_shots_on_target,away_shots_on_target,home_possession,away_possession,home_passes,away_passes,home_corners,away_corners,winner,season
0,mt_012211017,2023-10-07T14:00:00.000Z,8,Manchester United,Brentford,2,1,1.43,0.95,21.0,...,8.0,3.0,64.0,36.0,557.0,326.0,7.0,5.0,home,2023_24
1,mt_012211028,2023-10-21T14:00:00.000Z,9,Newcastle United,Crystal Palace,4,0,2.39,1.21,10.0,...,7.0,3.0,60.0,40.0,573.0,389.0,6.0,8.0,home,2023_24
2,mt_012211041,2023-10-29T14:00:00.000Z,10,Liverpool,Nottingham Forest,3,0,3.19,0.45,21.0,...,8.0,1.0,73.0,27.0,728.0,266.0,8.0,3.0,home,2023_24
3,mt_012211070,2023-09-30T16:30:00.000Z,7,Tottenham Hotspur,Liverpool,2,1,2.23,1.31,24.0,...,8.0,4.0,65.0,35.0,662.0,364.0,11.0,5.0,home,2023_24
4,mt_012211208,2023-11-04T15:00:00.000Z,11,Brentford,West Ham United,3,2,1.63,1.82,16.0,...,4.0,2.0,59.0,41.0,560.0,373.0,4.0,3.0,home,2023_24


In [ ]:
# Checking to see if there are any null values to clean up.
historical_df.isnull().sum()

match_id                0
date                    0
matchday                0
home_team               0
away_team               0
home_goals              0
away_goals              0
home_xg                 2
away_xg                 2
home_shots              2
away_shots              2
home_shots_on_target    3
away_shots_on_target    3
home_possession         2
away_possession         2
home_passes             3
away_passes             3
home_corners            2
away_corners            2
winner                  0
season                  0
dtype: int64

In [203]:
historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc = True
)

historical_df.dtypes

match_id                                str
date                    datetime64[us, UTC]
matchday                              int64
home_team                               str
away_team                               str
home_goals                            int64
away_goals                            int64
home_xg                             float64
away_xg                             float64
home_shots                          float64
away_shots                          float64
home_shots_on_target                float64
away_shots_on_target                float64
home_possession                     float64
away_possession                     float64
home_passes                         float64
away_passes                         float64
home_corners                        float64
away_corners                        float64
winner                                  str
season                                  str
dtype: object

In [204]:
# Conduct a final quality check.

print("Total matches:", len(historical_df))
print("Duplicate match IDs:", historical_df["match_id"].duplicated().sum())

print("\nMatches per season:")
print(historical_df["season"].value_counts())

print("\nDate range:")
print(historical_df["date"].min())
print(historical_df["date"].max())

Total matches: 1140
Duplicate match IDs: 0

Matches per season:
season
2023_24    380
2024_25    380
2025_26    380
Name: count, dtype: int64

Date range:
2023-08-11 19:00:00+00:00
2026-05-24 15:00:00+00:00


In [318]:
# Saving the processed historical dataset.

historical_file = PROCESSED_DIR / "historical_matches.csv"

historical_df.to_csv(
    historical_file,
    index = False
)

print("Saved to:", historical_file)

Saved to: ..\data\processed\historical_matches.csv


In [206]:
# Inspect a historical match's odds structure
# Will help us decide what fields to extract
# Shows us how the API organizes bookmakers, markets, and prices.
sample_file = next(SEASON_DIRS["2023_24"].glob("*.json"))

with open(sample_file, "r", encoding = "utf-8") as f:
    sample_match = json.load(f)

print(json.dumps(sample_match["odds"], indent = 2)[:5000])

{
  "data": {
    "match_id": "mt_012211017",
    "bookmakers": [
      {
        "bookmaker": "Bet365",
        "markets": {
          "match_odds": {
            "home": {
              "opening": null,
              "last_seen": "1.600"
            },
            "draw": {
              "opening": null,
              "last_seen": "4.330"
            },
            "away": {
              "opening": null,
              "last_seen": "5.000"
            }
          },
          "btts": {
            "yes": {
              "opening": null,
              "last_seen": "1.700"
            },
            "no": {
              "opening": null,
              "last_seen": "2.050"
            }
          },
          "total_goals": {
            "0.5": {
              "over": {
                "opening": null,
                "last_seen": "1.040"
              },
              "under": {
                "opening": null,
                "last_seen": "13.000"
              }
            },
      

In [207]:
# Check if all 1,140 matches have Bet365 odds
# Showing which bookmakers are available across the dataset and how often they appear.
bookmaker_counts = {}

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding = "utf-8") as f:
            match_data = json.load(f)

        bookmakers = match_data["odds"]["data"]["bookmakers"]

        for bookmaker in bookmakers:
            name = bookmaker["bookmaker"]

            bookmaker_counts[name] = bookmaker_counts.get(name, 0) + 1

print(bookmaker_counts)


{'Bet365': 1140}


In [208]:
# Make sure match odds are present for all 1,140 matches.

missing_match_odds = []

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding = "utf-8") as f:
            match_data = json.load(f)

        details = match_data["details"]["data"]
        bookmakers = match_data["odds"]["data"]["bookmakers"]

        bet365 = bookmakers[0]
        match_odds = bet365["markets"].get("match_odds")

        if match_odds is None:
            missing_match_odds.append({
                "season": season_name,
                "match_id": details["id"],
                "home_team": details["home_team"]["name"],
                "away_team": details["away_team"]["name"]
            })

print("Matches missing match odds:", len(missing_match_odds))

Matches missing match odds: 0


In [209]:
# Adding home/draw/away odds to our historical dataset.

odds_rows = []

for season_name in ["2023_24", "2024_25", "2025_26"]:
    season_dir = SEASON_DIRS[season_name]

    for file_path in season_dir.glob("*.json"):
        with open(file_path, "r", encoding="utf-8") as f:
            match_data = json.load(f)

        match_id = match_data["details"]["data"]["id"]

        bookmakers = match_data["odds"]["data"]["bookmakers"]
        match_odds = bookmakers[0]["markets"]["match_odds"]

        odds_rows.append({
            "match_id": match_id,
            "home_odds": float(match_odds["home"]["last_seen"]),
            "draw_odds": float(match_odds["draw"]["last_seen"]),
            "away_odds": float(match_odds["away"]["last_seen"])
        })

odds_df = pd.DataFrame(odds_rows)

odds_df.shape

(1140, 4)

In [210]:
# Merging into our DataFrame

historical_df = historical_df.merge(
    odds_df,
    on = "match_id",
    how = "left"
)

historical_df.shape

(1140, 24)

In [211]:
historical_df[
    [
        "home_team",
        "away_team",
        "home_odds",
        "draw_odds",
        "away_odds"
    ]
].head()

,home_team,away_team,home_odds,draw_odds,away_odds
0,Manchester United,Brentford,1.60,4.33,5.0
1,Newcastle United,Crystal Palace,1.50,4.33,6.5
2,Liverpool,Nottingham Forest,1.22,7.00,11.0
3,Tottenham Hotspur,Liverpool,3.00,3.80,2.2
4,Brentford,West Ham United,2.05,3.60,3.5


In [212]:
# Make sure none of the odds became missing after the merge.

historical_df[
    ["home_odds", "draw_odds", "away_odds"]
].isnull().sum()

home_odds    0
draw_odds    0
away_odds    0
dtype: int64

In [320]:
# Overwrite our historical matches CSV with updated DataFrame

historical_df.to_csv(
    historical_file,
    index = False 
)

print("Saved to:", historical_file)
print("Shape:", historical_df.shape)

Saved to: ..\data\processed\historical_matches.csv
Shape: (10653, 22)


# DATA PIPELINE PART 2
ADDING MORE COMPETITIONS

In [98]:
# Set up Championship seasons and create a directory to store these matches

CHAMPIONSHIP_ID = "comp_8321"

CHAMPIONSHIP_SEASONS = {
    "2026_27": "sn_3014533",
    "2025_26": "sn_3064530",
    "2024_25": "sn_2930227",
    "2023_24": "sn_343481",
    "2022_23": "sn_286965",
    "2021_22": "sn_033091",
    "2020_21": "sn_855998"
}

CHAMPIONSHIP_DIR = RAW_MATCH_DIR / "championship"

CHAMPIONSHIP_SEASON_DIRS = {}

for season_name in CHAMPIONSHIP_SEASONS:
    season_dir = CHAMPIONSHIP_DIR / season_name
    season_dir.mkdir(parents = True, exist_ok = True)

    CHAMPIONSHIP_SEASON_DIRS[season_name] = season_dir

print(CHAMPIONSHIP_SEASON_DIRS)

{'2026_27': WindowsPath('../data/raw/matches/championship/2026_27'), '2025_26': WindowsPath('../data/raw/matches/championship/2025_26'), '2024_25': WindowsPath('../data/raw/matches/championship/2024_25'), '2023_24': WindowsPath('../data/raw/matches/championship/2023_24'), '2022_23': WindowsPath('../data/raw/matches/championship/2022_23'), '2021_22': WindowsPath('../data/raw/matches/championship/2021_22'), '2020_21': WindowsPath('../data/raw/matches/championship/2020_21')}


In [99]:
# Test on one Championship season
championship_2022_23_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2022_23"]
)

print("Total matches returned:", len(championship_2022_23_matches))

Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5
Rate limit reached. Waiting 20 seconds before retry 3/5
Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5
Rate limit reached. Waiting 20 seconds before retry 3/5
Total matches returned: 557


In [100]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in championship_2022_23_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in championship_2022_23_matches))

Statuses:
Counter({'finished': 557})

Stages:
Counter({None: 552, 'semi_final': 4, 'final': 1})


In [101]:
# Check if the data is compatible

test_championship_match = championship_2022_23_matches[0]

print(
    test_championship_match["home_team"]["name"],
    "vs",
    test_championship_match["away_team"]["name"]
)

test_championship_data = get_baseline_match_data(
    test_championship_match["id"]
)

print(test_championship_data.keys())

Coventry City vs Luton Town
dict_keys(['details', 'statistics', 'odds'])


In [102]:
# Make a Championship save path
# Don't want to mix up the JSONs 

def collect_championship_baseline_data(season_name, matches):
    season_dir = CHAMPIONSHIP_SEASON_DIRS[season_name]

    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(
            match_id,
            match_data,
            season_dir
        )

        time.sleep(1)

In [103]:
# Collect 2022/23 Championship matches

collect_championship_baseline_data(
    "2022_23",
    championship_2022_23_matches
)

1/557 - Collecting: mt_581874371
2/557 - Collecting: mt_406526716
3/557 - Collecting: mt_367048338
4/557 - Collecting: mt_978638174
5/557 - Collecting: mt_624104353
Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5
Rate limit reached. Waiting 20 seconds before retry 3/5
6/557 - Collecting: mt_360165605
7/557 - Collecting: mt_259753983
8/557 - Collecting: mt_193694308
9/557 - Collecting: mt_976270602
Rate limit reached. Waiting 5 seconds before retry 1/5
10/557 - Collecting: mt_193694345
11/557 - Collecting: mt_405809942
12/557 - Collecting: mt_621928855
13/557 - Collecting: mt_360165581
Rate limit reached. Waiting 5 seconds before retry 1/5
Rate limit reached. Waiting 10 seconds before retry 2/5
Rate limit reached. Waiting 20 seconds before retry 3/5
14/557 - Collecting: mt_621928833
15/557 - Collecting: mt_587486627
16/557 - Collecting: mt_834032235
17/557 - Collecting: mt_742541129
Rate limit reached. Waiting 5 seconds befo

In [107]:
saved_championship_2021_22 = list(
    CHAMPIONSHIP_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved Championship 2021/22 files:", len(saved_championship_2021_22))

Saved Championship 2021/22 files: 536


In [111]:
championship_2020_21_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2020_21"]
)

print("Total matches returned:", len(championship_2020_21_matches))

Total matches returned: 557


In [113]:
# Collect 2020_21 Championship season

collect_championship_baseline_data(
    "2020_21",
    championship_2020_21_matches
)

1/557 - Skipping existing: mt_575521629
2/557 - Skipping existing: mt_131138187
3/557 - Skipping existing: mt_848848362
4/557 - Skipping existing: mt_727724757
5/557 - Skipping existing: mt_080081324
6/557 - Skipping existing: mt_452112657
7/557 - Skipping existing: mt_085995157
8/557 - Skipping existing: mt_452112074
9/557 - Skipping existing: mt_085995184
10/557 - Skipping existing: mt_138778311
11/557 - Skipping existing: mt_610550163
12/557 - Skipping existing: mt_304224080
13/557 - Skipping existing: mt_085995702
14/557 - Skipping existing: mt_963443072
15/557 - Skipping existing: mt_579333537
16/557 - Skipping existing: mt_085999554
17/557 - Skipping existing: mt_579333985
18/557 - Skipping existing: mt_575332523
19/557 - Skipping existing: mt_304224751
20/557 - Skipping existing: mt_452112040
21/557 - Skipping existing: mt_085995145
22/557 - Skipping existing: mt_847667409
23/557 - Skipping existing: mt_847667287
24/557 - Skipping existing: mt_304224570
25/557 - Skipping existin

In [114]:
saved_championship_2020_21 = list(
    CHAMPIONSHIP_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved Championship 2020/21 files:", len(saved_championship_2020_21))

Saved Championship 2020/21 files: 557


In [115]:
# Collect 2023/24 season

championship_2023_24_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2023_24"]
)

print("Total matches returned:", len(championship_2023_24_matches))

Total matches returned: 557


In [116]:
collect_championship_baseline_data(
    "2023_24",
    championship_2023_24_matches
)

1/557 - Collecting: mt_584868233
2/557 - Collecting: mt_196356808
3/557 - Collecting: mt_257907162
4/557 - Collecting: mt_972682302
5/557 - Collecting: mt_584714963
6/557 - Collecting: mt_978814673
7/557 - Collecting: mt_743390249
8/557 - Collecting: mt_367782066
9/557 - Collecting: mt_581123779
10/557 - Collecting: mt_978814664
11/557 - Collecting: mt_250068997
12/557 - Collecting: mt_195507330
13/557 - Collecting: mt_012249875
14/557 - Collecting: mt_195507345
15/557 - Collecting: mt_743390214
16/557 - Collecting: mt_839956424
17/557 - Collecting: mt_367782055
18/557 - Collecting: mt_407408711
19/557 - Collecting: mt_408998442
20/557 - Collecting: mt_012249848
21/557 - Collecting: mt_367782011
22/557 - Collecting: mt_581123747
23/557 - Collecting: mt_978814620
24/557 - Collecting: mt_195507308
25/557 - Collecting: mt_839956450
26/557 - Collecting: mt_624435134
27/557 - Collecting: mt_581123726
28/557 - Collecting: mt_195507357
29/557 - Collecting: mt_839956495
30/557 - Collecting: mt

In [117]:
saved_championship_2023_24 = list(
    CHAMPIONSHIP_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved Championship 2023/24 files:", len(saved_championship_2023_24))

Saved Championship 2023/24 files: 557


In [118]:
# Collect 2024/25 Championship season
championship_2024_25_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2024_25"]
)

print("Total matches returned:", len(championship_2024_25_matches))

Total matches returned: 559


In [119]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in championship_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in championship_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in championship_2024_25_matches
)))

Statuses:
Counter({'finished': 557, 'postponed': 2})

Stages:
Counter({None: 554, 'semi_final': 4, 'final': 1})

Unique IDs: 559


In [120]:
finished_championship_2024_25 = [
    match for match in championship_2024_25_matches
    if match["status"] == "finished"
]

print("Finished matches:", len(finished_championship_2024_25))

Finished matches: 557


In [121]:
collect_championship_baseline_data(
    "2024_25",
    finished_championship_2024_25
)

1/557 - Collecting: mt_838435709
2/557 - Collecting: mt_363510466
3/557 - Collecting: mt_838204739
4/557 - Collecting: mt_404985209
5/557 - Collecting: mt_626891021
6/557 - Collecting: mt_830528275
7/557 - Collecting: mt_361853547
8/557 - Collecting: mt_629386802
9/557 - Collecting: mt_745917109
10/557 - Collecting: mt_629386898
11/557 - Collecting: mt_584265623
12/557 - Collecting: mt_972109012
13/557 - Collecting: mt_745917194
14/557 - Collecting: mt_629386846
15/557 - Collecting: mt_013470729
16/557 - Collecting: mt_584265614
17/557 - Collecting: mt_972109081
18/557 - Collecting: mt_364473134
19/557 - Collecting: mt_257632311
20/557 - Collecting: mt_196041487
21/557 - Collecting: mt_408794925
22/557 - Collecting: mt_584265635
23/557 - Collecting: mt_257632388
24/557 - Collecting: mt_196041476
25/557 - Collecting: mt_361853526
26/557 - Collecting: mt_629386851
27/557 - Collecting: mt_830528269
28/557 - Collecting: mt_408794919
29/557 - Collecting: mt_013470730
30/557 - Collecting: mt

In [122]:
saved_championship_2024_25 = list(
    CHAMPIONSHIP_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved Championship 2024/25 files:", len(saved_championship_2024_25))

Saved Championship 2024/25 files: 557


In [124]:
# Collect 2025/26 Championship season

championship_2025_26_matches = get_all_season_matches(
    CHAMPIONSHIP_ID,
    CHAMPIONSHIP_SEASONS["2025_26"]
)

print("Total matches returned:", len(championship_2025_26_matches))

Total matches returned: 558


In [125]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in championship_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in championship_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in championship_2025_26_matches
)))

Statuses:
Counter({'finished': 557, 'postponed': 1})

Stages:
Counter({None: 552, 'semi_final': 4, 'final': 2})

Unique IDs: 558


In [126]:
finished_championship_2025_26 = [
    match for match in championship_2025_26_matches
    if match["status"] == "finished"
]

print("Finished matches:", len(finished_championship_2025_26))

Finished matches: 557


In [129]:
collect_championship_baseline_data(
    "2025_26",
    finished_championship_2025_26
)

1/557 - Skipping existing: mt_899557794
2/557 - Skipping existing: mt_466047116
3/557 - Skipping existing: mt_377638221
4/557 - Skipping existing: mt_155910486
5/557 - Skipping existing: mt_733479169
6/557 - Skipping existing: mt_747917034
7/557 - Skipping existing: mt_838528694
8/557 - Skipping existing: mt_363853275
9/557 - Skipping existing: mt_626386526
10/557 - Skipping existing: mt_404794102
11/557 - Skipping existing: mt_010470919
12/557 - Skipping existing: mt_585265384
13/557 - Skipping existing: mt_979109471
14/557 - Skipping existing: mt_252632850
15/557 - Skipping existing: mt_191041799
16/557 - Skipping existing: mt_747917042
17/557 - Skipping existing: mt_363853203
18/557 - Skipping existing: mt_734524246
19/557 - Skipping existing: mt_010470983
20/557 - Skipping existing: mt_838528632
21/557 - Skipping existing: mt_626386510
22/557 - Skipping existing: mt_979109468
23/557 - Skipping existing: mt_252632895
24/557 - Skipping existing: mt_363853254
25/557 - Skipping existin

In [130]:
for season in [
    "2020_21",
    "2021_22",
    "2022_23",
    "2023_24",
    "2024_25",
    "2025_26"
]:
    files = list(CHAMPIONSHIP_SEASON_DIRS[season].glob("*.json"))
    print(season, len(files))

2020_21 557
2021_22 557
2022_23 557
2023_24 557
2024_25 557
2025_26 557


In [131]:
# Adding Champions League Seasons

CHAMPIONS_LEAGUE_ID = "comp_3498"

CHAMPIONS_LEAGUE_SEASONS = {
    "2026_27": "sn_8406903",
    "2025_26": "sn_5783461",
    "2024_25": "sn_0872372",
    "2023_24": "sn_939193",
    "2022_23": "sn_327866",
    "2021_22": "sn_699943",
    "2020_21": "sn_388063"
}

In [132]:
# Create a folder for Champions League matches

CHAMPIONS_LEAGUE_DIR = RAW_MATCH_DIR / "champions_league"

CHAMPIONS_LEAGUE_SEASON_DIRS = {}

for season_name in CHAMPIONS_LEAGUE_SEASONS:
    season_dir = CHAMPIONS_LEAGUE_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)

    CHAMPIONS_LEAGUE_SEASON_DIRS[season_name] = season_dir

print(CHAMPIONS_LEAGUE_SEASON_DIRS)

{'2026_27': WindowsPath('../data/raw/matches/champions_league/2026_27'), '2025_26': WindowsPath('../data/raw/matches/champions_league/2025_26'), '2024_25': WindowsPath('../data/raw/matches/champions_league/2024_25'), '2023_24': WindowsPath('../data/raw/matches/champions_league/2023_24'), '2022_23': WindowsPath('../data/raw/matches/champions_league/2022_23'), '2021_22': WindowsPath('../data/raw/matches/champions_league/2021_22'), '2020_21': WindowsPath('../data/raw/matches/champions_league/2020_21')}


In [ ]:
# Inspect 2023/24 season (Before format change)

ucl_2023_24_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2023_24"]
)

print("Total matches returned:", len(ucl_2023_24_matches))

Total matches returned: 214


In [134]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2023_24_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2023_24_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2023_24_matches
)))

Statuses:
Counter({'finished': 214})

Stages:
Counter({None: 96, 'qualifying': 74, 'round_of_16': 28, 'quarter_final': 8, 'semi_final': 6, 'final': 2})

Unique IDs: 214


In [135]:
stage_examples = {}

for match in ucl_2023_24_matches:
    stage = match.get("stage_name")

    if stage not in stage_examples:
        stage_examples[stage] = match

for stage, match in stage_examples.items():
    print(
        stage,
        "-",
        match["home_team"]["name"],
        "vs",
        match["away_team"]["name"]
    )

final - Borussia Dortmund vs Real Madrid
semi_final - Real Madrid vs FC Bayern München
quarter_final - FC Bayern München vs Arsenal
round_of_16 - Atlético Madrid vs Inter
None - Borussia Dortmund vs Paris Saint-Germain
qualifying - AEK Athens vs GNK Dinamo Zagreb


In [136]:
# Check for Unique IDs

for stage in ["round_of_16", "semi_final", "final"]:
    print(f"\n--- {stage.upper()} ---")

    stage_matches = [
        match for match in ucl_2023_24_matches
        if match.get("stage_name") == stage
    ]

    for match in sorted(stage_matches, key=lambda x: x["utc_date"]):
        print(
            match["utc_date"][:10],
            match["id"],
            match["home_team"]["name"],
            "vs",
            match["away_team"]["name"]
        )


--- ROUND_OF_16 ---
2023-08-22 mt_195770686 Royal Antwerp FC vs AEK Athens
2023-08-22 mt_581327334 Raków Częstochowa vs FC København
2023-08-22 mt_978416441 Rangers vs PSV Eindhoven
2023-08-23 mt_012948955 Sporting Braga vs Panathinaikos
2023-08-23 mt_978416432 Molde FK vs Galatasaray
2023-08-23 mt_743092064 Maccabi Haifa vs Young Boys
2023-08-29 mt_406170250 Young Boys vs Maccabi Haifa
2023-08-29 mt_978417309 Galatasaray vs Molde FK
2023-08-29 mt_250865131 Panathinaikos vs Sporting Braga
2023-08-30 mt_250886718 AEK Athens vs Royal Antwerp FC
2023-08-30 mt_250869816 PSV Eindhoven vs Rangers
2023-08-30 mt_195703785 FC København vs Raków Częstochowa
2024-02-13 mt_749359750 RB Leipzig vs Real Madrid
2024-02-13 mt_407687480 FC København vs Manchester City
2024-02-14 mt_971821919 Lazio vs FC Bayern München
2024-02-14 mt_835905855 Paris Saint-Germain vs Real Sociedad
2024-02-20 mt_190560107 PSV Eindhoven vs Borussia Dortmund
2024-02-20 mt_749359795 Inter vs Atlético Madrid
2024-02-21 mt_368

In [138]:
# Create a collector for Champions League seasons

def collect_champions_league_baseline_data(season_name, matches):
    season_dir = CHAMPIONS_LEAGUE_SEASON_DIRS[season_name]

    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(
            match_id,
            match_data,
            season_dir
        )

        time.sleep(1)
 

In [139]:
# Collect 2023_24 CL season

collect_champions_league_baseline_data(
    "2023_24",
    ucl_2023_24_matches
)

1/214 - Collecting: mt_749723742
2/214 - Collecting: mt_835849832
3/214 - Collecting: mt_368307303
4/214 - Collecting: mt_407456451
5/214 - Collecting: mt_623614610
6/214 - Collecting: mt_014082083
7/214 - Collecting: mt_971968968
8/214 - Collecting: mt_256290295
9/214 - Collecting: mt_749676092
10/214 - Collecting: mt_190135133
11/214 - Collecting: mt_256121860
12/214 - Collecting: mt_190818709
13/214 - Collecting: mt_835787652
14/214 - Collecting: mt_749359767
15/214 - Collecting: mt_368718342
16/214 - Collecting: mt_407687448
17/214 - Collecting: mt_971821997
18/214 - Collecting: mt_407687427
19/214 - Collecting: mt_014234052
20/214 - Collecting: mt_256076229
21/214 - Collecting: mt_623493609
22/214 - Collecting: mt_368718387
23/214 - Collecting: mt_623493632
24/214 - Collecting: mt_190560107
25/214 - Collecting: mt_749359795
26/214 - Collecting: mt_971821919
27/214 - Collecting: mt_835905855
28/214 - Collecting: mt_749359750
29/214 - Collecting: mt_407687480
30/214 - Collecting: mt

In [140]:
saved_ucl_2023_24 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved UCL 2023/24 files:", len(saved_ucl_2023_24))

Saved UCL 2023/24 files: 214


In [141]:
# Collect 2022/23 CL season

ucl_2022_23_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2022_23"]
)

print("Total matches returned:", len(ucl_2022_23_matches))

Total matches returned: 214


In [142]:
collect_champions_league_baseline_data(
    "2022_23",
    ucl_2022_23_matches
)

1/214 - Collecting: mt_012183397
2/214 - Collecting: mt_367054855
3/214 - Collecting: mt_624180554
4/214 - Collecting: mt_195347766
5/214 - Collecting: mt_195347788
6/214 - Collecting: mt_011034475
7/214 - Collecting: mt_977921102
8/214 - Collecting: mt_588542263
9/214 - Collecting: mt_255276636
10/214 - Collecting: mt_744759914
11/214 - Collecting: mt_366318855
12/214 - Collecting: mt_199160045
13/214 - Collecting: mt_833805524
14/214 - Collecting: mt_833524407
15/214 - Collecting: mt_622381199
16/214 - Collecting: mt_199043374
17/214 - Collecting: mt_011478832
18/214 - Collecting: mt_366850012
19/214 - Collecting: mt_255639989
20/214 - Collecting: mt_744912257
21/214 - Collecting: mt_400795587
22/214 - Collecting: mt_977106626
23/214 - Collecting: mt_622381133
24/214 - Collecting: mt_199043301
25/214 - Collecting: mt_833524456
26/214 - Collecting: mt_255639973
27/214 - Collecting: mt_366850081
28/214 - Collecting: mt_588267748
29/214 - Collecting: mt_744912296
30/214 - Collecting: mt

In [143]:
saved_ucl_2022_23 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved UCL 2022/23 files:", len(saved_ucl_2022_23))

Saved UCL 2022/23 files: 214


In [144]:
# Collect 2021/22 CL season

ucl_2021_22_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2021_22"]
)

print("Total matches returned:", len(ucl_2021_22_matches))

Total matches returned: 218


In [145]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2021_22_matches
)))

Statuses:
Counter({'finished': 218})

Stages:
Counter({None: 96, 'qualifying': 90, 'round_of_16': 16, 'quarter_final': 8, 'semi_final': 6, 'final': 2})

Unique IDs: 218


In [146]:
collect_champions_league_baseline_data(
    "2021_22",
    ucl_2021_22_matches
)

1/218 - Collecting: mt_193598476
2/218 - Collecting: mt_587849297
3/218 - Collecting: mt_742456900
4/218 - Collecting: mt_834307560
5/218 - Collecting: mt_018135451
6/218 - Collecting: mt_018899922
7/218 - Collecting: mt_834466636
8/218 - Collecting: mt_405511106
9/218 - Collecting: mt_018899911
10/218 - Collecting: mt_360022261
11/218 - Collecting: mt_587733387
12/218 - Collecting: mt_193377791
13/218 - Collecting: mt_742200046
14/218 - Collecting: mt_586946873
15/218 - Collecting: mt_253173596
16/218 - Collecting: mt_017537185
17/218 - Collecting: mt_832702344
18/218 - Collecting: mt_832702333
19/218 - Collecting: mt_628098228
20/218 - Collecting: mt_365415660
21/218 - Collecting: mt_628098286
22/218 - Collecting: mt_365415605
23/218 - Collecting: mt_017537179
24/218 - Collecting: mt_194864935
25/218 - Collecting: mt_409289092
26/218 - Collecting: mt_741651443
27/218 - Collecting: mt_741651424
28/218 - Collecting: mt_409289054
29/218 - Collecting: mt_970320762
30/218 - Collecting: mt

In [147]:
saved_ucl_2021_22 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved UCL 2021/22 files:", len(saved_ucl_2021_22))

Saved UCL 2021/22 files: 218


In [148]:
# Collect 2020/21 CL season

ucl_2020_21_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2020_21"]
)

print("Total matches returned:", len(ucl_2020_21_matches))

Total matches returned: 177


In [149]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2020_21_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2020_21_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2020_21_matches
)))

Statuses:
Counter({'finished': 177})

Stages:
Counter({None: 96, 'qualifying': 49, 'round_of_16': 16, 'quarter_final': 8, 'semi_final': 6, 'final': 2})

Unique IDs: 177


In [150]:
collect_champions_league_baseline_data(
    "2020_21",
    ucl_2020_21_matches
)

1/177 - Collecting: mt_080078873
2/177 - Collecting: mt_454266877
3/177 - Collecting: mt_616044939
4/177 - Collecting: mt_575911428
5/177 - Collecting: mt_080522342
6/177 - Collecting: mt_454168465
7/177 - Collecting: mt_969482973
8/177 - Collecting: mt_080923027
9/177 - Collecting: mt_575314585
10/177 - Collecting: mt_848690895
11/177 - Collecting: mt_303271377
12/177 - Collecting: mt_727035735
13/177 - Collecting: mt_616549642
14/177 - Collecting: mt_131554189
15/177 - Collecting: mt_727331762
16/177 - Collecting: mt_080227059
17/177 - Collecting: mt_848992872
18/177 - Collecting: mt_454669422
19/177 - Collecting: mt_292003210
20/177 - Collecting: mt_575116594
21/177 - Collecting: mt_969880931
22/177 - Collecting: mt_080227093
23/177 - Collecting: mt_131554173
24/177 - Collecting: mt_969880948
25/177 - Collecting: mt_727331701
26/177 - Collecting: mt_303775323
27/177 - Collecting: mt_616448650
28/177 - Collecting: mt_454669411
29/177 - Collecting: mt_575116532
30/177 - Collecting: mt

In [151]:
saved_ucl_2020_21 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved UCL 2020/21 files:", len(saved_ucl_2020_21))

Saved UCL 2020/21 files: 177


In [152]:
# Collect 2024/25 CL season (new format)

ucl_2024_25_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2024_25"]
)

print("Total matches returned:", len(ucl_2024_25_matches))

Total matches returned: 279


In [153]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2024_25_matches
)))

Statuses:
Counter({'finished': 279})

Stages:
Counter({None: 144, 'qualifying': 76, 'round_of_16': 46, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 279


In [156]:
collect_champions_league_baseline_data(
    "2024_25",
    ucl_2024_25_matches
)

1/279 - Skipping existing: mt_015208290
2/279 - Skipping existing: mt_364741364
3/279 - Skipping existing: mt_620409625
4/279 - Skipping existing: mt_015253014
5/279 - Skipping existing: mt_402628408
6/279 - Skipping existing: mt_973832988
7/279 - Skipping existing: mt_198586153
8/279 - Skipping existing: mt_251017205
9/279 - Skipping existing: mt_402628457
10/279 - Skipping existing: mt_620409619
11/279 - Skipping existing: mt_015253082
12/279 - Skipping existing: mt_589194578
13/279 - Skipping existing: mt_973832966
14/279 - Skipping existing: mt_837979789
15/279 - Skipping existing: mt_015252550
16/279 - Skipping existing: mt_198585880
17/279 - Skipping existing: mt_973838334
18/279 - Skipping existing: mt_973838393
19/279 - Skipping existing: mt_198585816
20/279 - Skipping existing: mt_620404061
21/279 - Skipping existing: mt_015252077
22/279 - Skipping existing: mt_589191955
23/279 - Skipping existing: mt_251010128
24/279 - Skipping existing: mt_402626249
25/279 - Skipping existin

In [157]:
saved_ucl_2024_25 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved UCL 2024/25 files:", len(saved_ucl_2024_25))

Saved UCL 2024/25 files: 279


In [158]:
# Collecting 2025/26 CL season

ucl_2025_26_matches = get_all_season_matches(
    CHAMPIONS_LEAGUE_ID,
    CHAMPIONS_LEAGUE_SEASONS["2025_26"]
)

print("Total matches returned:", len(ucl_2025_26_matches))

Total matches returned: 281


In [159]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in ucl_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in ucl_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in ucl_2025_26_matches
)))

Statuses:
Counter({'finished': 281})

Stages:
Counter({None: 144, 'qualifying': 78, 'round_of_16': 46, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 281


In [160]:
collect_champions_league_baseline_data(
    "2025_26",
    ucl_2025_26_matches
)

1/281 - Collecting: mt_642459639
2/281 - Collecting: mt_021293042
3/281 - Collecting: mt_460618477
4/281 - Collecting: mt_518134528
5/281 - Collecting: mt_987842916
6/281 - Collecting: mt_205087263
7/281 - Collecting: mt_893960466
8/281 - Collecting: mt_734305206
9/281 - Collecting: mt_376721021
10/281 - Collecting: mt_460618516
11/281 - Collecting: mt_518134737
12/281 - Collecting: mt_642459153
13/281 - Collecting: mt_021293891
14/281 - Collecting: mt_205086923
15/281 - Collecting: mt_159570381
16/281 - Collecting: mt_734309266
17/281 - Collecting: mt_893965476
18/281 - Collecting: mt_987841696
19/281 - Collecting: mt_376728041
20/281 - Collecting: mt_460617526
21/281 - Collecting: mt_642453103
22/281 - Collecting: mt_205086968
23/281 - Collecting: mt_460617579
24/281 - Collecting: mt_734309233
25/281 - Collecting: mt_518132797
26/281 - Collecting: mt_205086982
27/281 - Collecting: mt_642453154
28/281 - Collecting: mt_376728017
29/281 - Collecting: mt_205086971
30/281 - Collecting: mt

In [161]:
saved_ucl_2025_26 = list(
    CHAMPIONS_LEAGUE_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved UCL 2025/26 files:", len(saved_ucl_2025_26))

Saved UCL 2025/26 files: 281


In [162]:
# Collecting FA cup seasons

FA_CUP_ID = "comp_7428"

FA_CUP_SEASONS = {
    "2026_27": "sn_7205142",
    "2025_26": "sn_3071665",
    "2024_25": "sn_8449488",
    "2023_24": "sn_885727",
    "2022_23": "sn_531226",
    "2021_22": "sn_097717",
    "2020_21": "sn_169084"
}

In [163]:
# Create separate folder to store FA cup matches

FA_CUP_DIR = RAW_MATCH_DIR / "fa_cup"

FA_CUP_SEASON_DIRS = {}

for season_name in FA_CUP_SEASONS:
    season_dir = FA_CUP_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    FA_CUP_SEASON_DIRS[season_name] = season_dir

In [164]:
# Collector for matches

def collect_fa_cup_baseline_data(season_name, matches):
    season_dir = FA_CUP_SEASON_DIRS[season_name]
    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(match_id, match_data, season_dir)
        time.sleep(1)

In [ ]:
# Collect 2023/24 FA Cup matches

fa_cup_2023_24_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2023_24"]
)

print("Total matches returned:", len(fa_cup_2023_24_matches))

Total matches returned: 149


In [166]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2023_24_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2023_24_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2023_24_matches
)))

Statuses:
Counter({'finished': 149})

Stages:
Counter({None: 142, 'quarter_final': 4, 'semi_final': 2, 'final': 1})

Unique IDs: 149


In [167]:
collect_fa_cup_baseline_data(
    "2023_24",
    fa_cup_2023_24_matches
)

1/149 - Collecting: mt_972030842
2/149 - Collecting: mt_368351478
3/149 - Collecting: mt_835820790
4/149 - Collecting: mt_256193788
5/149 - Collecting: mt_971360243
6/149 - Collecting: mt_582976435
7/149 - Collecting: mt_190834676
8/149 - Collecting: mt_256837517
9/149 - Collecting: mt_835642455
10/149 - Collecting: mt_582364176
11/149 - Collecting: mt_190734356
12/149 - Collecting: mt_582364899
13/149 - Collecting: mt_014973288
14/149 - Collecting: mt_971402734
15/149 - Collecting: mt_971402809
16/149 - Collecting: mt_014341798
17/149 - Collecting: mt_623932854
18/149 - Collecting: mt_014342208
Request failed: 404
{"error":{"code":"NOT_FOUND","message":"Match statistics not found","status_code":404}}
Saving with missing sections: ['statistics']
19/149 - Collecting: mt_407870910
20/149 - Collecting: mt_582428636
Request failed: 404
{"error":{"code":"NOT_FOUND","message":"Match statistics not found","status_code":404}}
Saving with missing sections: ['statistics']
21/149 - Collecting: mt

In [168]:
saved_fa_cup_2023_24 = list(
    FA_CUP_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved FA Cup 2023/24 files:", len(saved_fa_cup_2023_24))

Saved FA Cup 2023/24 files: 149


In [169]:
# Collect 2022/23 FA cup season

fa_cup_2022_23_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2022_23"]
)

print("Total matches returned:", len(fa_cup_2022_23_matches))

Total matches returned: 144


In [170]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2022_23_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2022_23_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2022_23_matches
)))

Statuses:
Counter({'finished': 144})

Stages:
Counter({None: 137, 'quarter_final': 4, 'semi_final': 2, 'final': 1})

Unique IDs: 144


In [171]:
collect_fa_cup_baseline_data(
    "2022_23",
    fa_cup_2022_23_matches
)

1/144 - Collecting: mt_012818277
2/144 - Collecting: mt_255280160
3/144 - Collecting: mt_588531566
4/144 - Collecting: mt_255125703
Request failed: 404
{"error":{"code":"NOT_FOUND","message":"Odds not found for this match","status_code":404}}
Saving with missing sections: ['odds']
5/144 - Collecting: mt_744674546
6/144 - Collecting: mt_199819691
7/144 - Collecting: mt_833783036
8/144 - Collecting: mt_977303986
9/144 - Collecting: mt_833668633
10/144 - Collecting: mt_977303994
11/144 - Collecting: mt_588969518
12/144 - Collecting: mt_744007043
13/144 - Collecting: mt_833744357
14/144 - Collecting: mt_199789108
15/144 - Collecting: mt_366223260
16/144 - Collecting: mt_977442663
17/144 - Collecting: mt_255888520
18/144 - Collecting: mt_011993817
19/144 - Collecting: mt_588334775
20/144 - Collecting: mt_255887998
21/144 - Collecting: mt_255886280
22/144 - Collecting: mt_744575504
23/144 - Collecting: mt_833080064
24/144 - Collecting: mt_366131125
25/144 - Collecting: mt_977479097
26/144 - 

In [172]:
saved_fa_cup_2022_23 = list(
    FA_CUP_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved FA Cup 2022/23 files:", len(saved_fa_cup_2022_23))

Saved FA Cup 2022/23 files: 144


In [173]:
# Collect 2021/22 FA cup season

fa_cup_2021_22_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2021_22"]
)

print("Total matches returned:", len(fa_cup_2021_22_matches))

Total matches returned: 136


In [174]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2021_22_matches
)))

Statuses:
Counter({'finished': 136})

Stages:
Counter({None: 129, 'quarter_final': 4, 'semi_final': 2, 'final': 1})

Unique IDs: 136


In [175]:
collect_fa_cup_baseline_data(
    "2021_22",
    fa_cup_2021_22_matches
)

1/136 - Collecting: mt_834948985
2/136 - Collecting: mt_193385684
3/136 - Collecting: mt_018854178
4/136 - Collecting: mt_405562665
5/136 - Collecting: mt_193396437
6/136 - Collecting: mt_834430407
7/136 - Collecting: mt_259957391
8/136 - Collecting: mt_259305261
9/136 - Collecting: mt_360577715
10/136 - Collecting: mt_587617536
11/136 - Collecting: mt_976086204
12/136 - Collecting: mt_259309737
13/136 - Collecting: mt_587617469
14/136 - Collecting: mt_405965859
15/136 - Collecting: mt_018728370
16/136 - Collecting: mt_365370166
17/136 - Collecting: mt_970986308
18/136 - Collecting: mt_970986191
19/136 - Collecting: mt_409465809
20/136 - Collecting: mt_194153696
21/136 - Collecting: mt_586517485
22/136 - Collecting: mt_832894039
23/136 - Collecting: mt_628641921
24/136 - Collecting: mt_017028380
25/136 - Collecting: mt_586517479
26/136 - Collecting: mt_970986264
27/136 - Collecting: mt_253209797
28/136 - Collecting: mt_586517254
29/136 - Collecting: mt_253209758
30/136 - Collecting: mt

In [176]:
saved_fa_cup_2021_22 = list(
    FA_CUP_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved FA Cup 2021/22 files:", len(saved_fa_cup_2021_22))

Saved FA Cup 2021/22 files: 136


In [177]:
# Collect 2020/21 FA cup season

fa_cup_2020_21_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2020_21"]
)

print("Total matches returned:", len(fa_cup_2020_21_matches))

Total matches returned: 123


In [178]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2020_21_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2020_21_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2020_21_matches
)))

Statuses:
Counter({'finished': 123})

Stages:
Counter({None: 116, 'quarter_final': 4, 'semi_final': 2, 'final': 1})

Unique IDs: 123


In [179]:
collect_fa_cup_baseline_data(
    "2020_21",
    fa_cup_2020_21_matches
)

1/123 - Collecting: mt_454289680
2/123 - Collecting: mt_080941822
3/123 - Collecting: mt_575328718
4/123 - Collecting: mt_080384858
5/123 - Collecting: mt_727529276
6/123 - Collecting: mt_575472796
7/123 - Collecting: mt_454857520
8/123 - Collecting: mt_080431273
9/123 - Collecting: mt_969114116
10/123 - Collecting: mt_616392964
11/123 - Collecting: mt_292675035
12/123 - Collecting: mt_080449442
13/123 - Collecting: mt_575223228
14/123 - Collecting: mt_969127808
15/123 - Collecting: mt_292676293
16/123 - Collecting: mt_303873654
17/123 - Collecting: mt_080488454
18/123 - Collecting: mt_131033013
19/123 - Collecting: mt_080488403
20/123 - Collecting: mt_848544578
21/123 - Collecting: mt_616311305
22/123 - Collecting: mt_303800844
23/123 - Collecting: mt_575277252
24/123 - Collecting: mt_454755440
25/123 - Collecting: mt_303800833
26/123 - Collecting: mt_616311360
27/123 - Collecting: mt_454755741
28/123 - Collecting: mt_969166198
29/123 - Collecting: mt_454755728
30/123 - Collecting: mt

In [180]:
saved_fa_cup_2020_21 = list(
    FA_CUP_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved FA Cup 2020/21 files:", len(saved_fa_cup_2020_21))

Saved FA Cup 2020/21 files: 123


In [181]:
# Collect 2024/25 FA cup season

fa_cup_2024_25_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2024_25"]
)

print("Total matches returned:", len(fa_cup_2024_25_matches))

Total matches returned: 123


In [182]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2024_25_matches
)))

Statuses:
Counter({'finished': 123})

Stages:
Counter({None: 116, 'quarter_final': 4, 'semi_final': 2, 'final': 1})

Unique IDs: 123


In [183]:
collect_fa_cup_baseline_data(
    "2024_25",
    fa_cup_2024_25_matches
)

1/123 - Collecting: mt_585663327
2/123 - Collecting: mt_364231276
3/123 - Collecting: mt_837680699
4/123 - Collecting: mt_198066538
5/123 - Collecting: mt_251677092
6/123 - Collecting: mt_251683870
7/123 - Collecting: mt_620399414
8/123 - Collecting: mt_015210289
9/123 - Collecting: mt_837377827
10/123 - Collecting: mt_198988134
11/123 - Collecting: mt_746466717
12/123 - Collecting: mt_620200689
13/123 - Collecting: mt_251522823
14/123 - Collecting: mt_364644352
15/123 - Collecting: mt_402022497
16/123 - Collecting: mt_973661498
17/123 - Collecting: mt_251996825
18/123 - Collecting: mt_589634324
19/123 - Collecting: mt_620859536
20/123 - Collecting: mt_015793949
21/123 - Collecting: mt_837260652
22/123 - Collecting: mt_402918172
23/123 - Collecting: mt_973042411
24/123 - Collecting: mt_402945817
25/123 - Collecting: mt_015708392
26/123 - Collecting: mt_251387860
27/123 - Collecting: mt_364521273
28/123 - Collecting: mt_620859540
29/123 - Collecting: mt_973096401
30/123 - Collecting: mt

In [184]:
saved_fa_cup_2024_25 = list(
    FA_CUP_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved FA Cup 2024/25 files:", len(saved_fa_cup_2024_25))

Saved FA Cup 2024/25 files: 123


In [185]:
# Collect 2025/26 FA cup season

fa_cup_2025_26_matches = get_all_season_matches(
    FA_CUP_ID,
    FA_CUP_SEASONS["2025_26"]
)

print("Total matches returned:", len(fa_cup_2025_26_matches))

Total matches returned: 123


In [186]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in fa_cup_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in fa_cup_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in fa_cup_2025_26_matches
)))

Statuses:
Counter({'finished': 123})

Stages:
Counter({None: 116, 'quarter_final': 4, 'semi_final': 2, 'final': 1})

Unique IDs: 123


In [187]:
collect_fa_cup_baseline_data(
    "2025_26",
    fa_cup_2025_26_matches
)

1/123 - Collecting: mt_155994774
2/123 - Collecting: mt_377560288
3/123 - Collecting: mt_644821534
4/123 - Collecting: mt_205617336
5/123 - Collecting: mt_159080065
6/123 - Collecting: mt_734969954
7/123 - Collecting: mt_893575504
8/123 - Collecting: mt_376648321
9/123 - Collecting: mt_734491561
10/123 - Collecting: mt_460794620
11/123 - Collecting: mt_893352068
12/123 - Collecting: mt_376685124
13/123 - Collecting: mt_205563715
14/123 - Collecting: mt_159904683
15/123 - Collecting: mt_518826492
16/123 - Collecting: mt_987768874
17/123 - Collecting: mt_986960210
18/123 - Collecting: mt_732721530
19/123 - Collecting: mt_894842090
20/123 - Collecting: mt_986960279
21/123 - Collecting: mt_641618944
22/123 - Collecting: mt_517576427
23/123 - Collecting: mt_986967877
24/123 - Collecting: mt_465459860
25/123 - Collecting: mt_986934793
26/123 - Collecting: mt_028087341
27/123 - Collecting: mt_028087328
28/123 - Collecting: mt_517576416
29/123 - Collecting: mt_153134658
30/123 - Collecting: mt

In [188]:
saved_fa_cup_2025_26 = list(
    FA_CUP_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved FA Cup 2025/26 files:", len(saved_fa_cup_2025_26))

Saved FA Cup 2025/26 files: 123


In [189]:
# Creating code to collect EFL Cup matches

EFL_CUP_ID = "comp_2504"

EFL_CUP_SEASONS = {
    "2026_27": "sn_4588470",
    "2025_26": "sn_1398983",
    "2024_25": "sn_5762314",
    "2023_24": "sn_595992",
    "2022_23": "sn_532755",
    "2021_22": "sn_311334",
    "2020_21": "sn_044278"
}

In [190]:
# Create separate folder for EFL Cup matches

EFL_CUP_DIR = RAW_MATCH_DIR / "efl_cup"

EFL_CUP_SEASON_DIRS = {}

for season_name in EFL_CUP_SEASONS:
    season_dir = EFL_CUP_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    EFL_CUP_SEASON_DIRS[season_name] = season_dir

In [191]:
# Collector for matches in EFL Cup

def collect_efl_cup_baseline_data(season_name, matches):
    season_dir = EFL_CUP_SEASON_DIRS[season_name]
    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(match_id, match_data, season_dir)
        time.sleep(1)

In [192]:
# Collect 2023/24 EFL Cup matches

efl_cup_2023_24_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2023_24"]
)

print("Total matches returned:", len(efl_cup_2023_24_matches))

Total matches returned: 93


In [193]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2023_24_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2023_24_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2023_24_matches
)))

Statuses:
Counter({'finished': 93})

Stages:
Counter({None: 84, 'semi_final': 4, 'quarter_final': 4, 'final': 1})

Unique IDs: 93


In [194]:
collect_efl_cup_baseline_data(
    "2023_24",
    efl_cup_2023_24_matches
)

1/93 - Collecting: mt_014314841
2/93 - Collecting: mt_368727166
3/93 - Collecting: mt_623454921
4/93 - Collecting: mt_971847978
5/93 - Collecting: mt_014291013
6/93 - Collecting: mt_582797565
7/93 - Collecting: mt_256919238
8/93 - Collecting: mt_014858087
9/93 - Collecting: mt_971636903
10/93 - Collecting: mt_971007023
11/93 - Collecting: mt_582668645
12/93 - Collecting: mt_256335378
13/93 - Collecting: mt_749114159
14/93 - Collecting: mt_835223209
15/93 - Collecting: mt_407990915
16/93 - Collecting: mt_014771797
17/93 - Collecting: mt_190449466
18/93 - Collecting: mt_250101395
19/93 - Collecting: mt_624040810
20/93 - Collecting: mt_406262951
21/93 - Collecting: mt_581919672
22/93 - Collecting: mt_195858433
23/93 - Collecting: mt_743636121
24/93 - Collecting: mt_839797228
25/93 - Collecting: mt_367474554
26/93 - Collecting: mt_624045603
27/93 - Collecting: mt_839797232
28/93 - Collecting: mt_743636142
29/93 - Collecting: mt_367474503
30/93 - Collecting: mt_012525783
31/93 - Collecting:

In [195]:
saved_efl_cup_2023_24 = list(
    EFL_CUP_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved EFL Cup 2023/24 files:", len(saved_efl_cup_2023_24))

Saved EFL Cup 2023/24 files: 93


In [196]:
# Collect 2022/23 EFL Cup season

efl_cup_2022_23_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2022_23"]
)

print("Total matches returned:", len(efl_cup_2022_23_matches))

Total matches returned: 93


In [197]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2022_23_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2022_23_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2022_23_matches
)))

Statuses:
Counter({'finished': 93})

Stages:
Counter({None: 84, 'semi_final': 4, 'quarter_final': 4, 'final': 1})

Unique IDs: 93


In [198]:
collect_efl_cup_baseline_data(
    "2022_23",
    efl_cup_2022_23_matches
)

1/93 - Collecting: mt_833677780
2/93 - Collecting: mt_833629068
3/93 - Collecting: mt_977408402
4/93 - Collecting: mt_622584922
5/93 - Collecting: mt_255830751
6/93 - Collecting: mt_833055324
7/93 - Collecting: mt_400877542
8/93 - Collecting: mt_366188655
9/93 - Collecting: mt_622933166
10/93 - Collecting: mt_588286114
11/93 - Collecting: mt_400709662
12/93 - Collecting: mt_366865631
13/93 - Collecting: mt_622328263
14/93 - Collecting: mt_622328446
15/93 - Collecting: mt_011417229
16/93 - Collecting: mt_977170881
17/93 - Collecting: mt_833532386
18/93 - Collecting: mt_833222649
19/93 - Collecting: mt_833222623
20/93 - Collecting: mt_622888588
21/93 - Collecting: mt_199444736
22/93 - Collecting: mt_744111029
23/93 - Collecting: mt_400999159
24/93 - Collecting: mt_011777970
25/93 - Collecting: mt_255333837
26/93 - Collecting: mt_199444740
27/93 - Collecting: mt_255333898
28/93 - Collecting: mt_366555206
29/93 - Collecting: mt_588666369
30/93 - Collecting: mt_744111013
31/93 - Collecting:

In [199]:
saved_efl_cup_2022_23 = list(
    EFL_CUP_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved EFL Cup 2022/23 files:", len(saved_efl_cup_2022_23))

Saved EFL Cup 2022/23 files: 93


In [200]:
# Collect 2021/22 FA Cup season

efl_cup_2021_22_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2021_22"]
)

print("Total matches returned:", len(efl_cup_2021_22_matches))

Total matches returned: 93


In [201]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2021_22_matches
)))

Statuses:
Counter({'finished': 93})

Stages:
Counter({None: 84, 'semi_final': 4, 'quarter_final': 4, 'final': 1})

Unique IDs: 93


In [202]:
collect_efl_cup_baseline_data(
    "2021_22",
    efl_cup_2021_22_matches
)

1/93 - Collecting: mt_253281021
2/93 - Collecting: mt_741742974
3/93 - Collecting: mt_365438832
4/93 - Collecting: mt_741679977
5/93 - Collecting: mt_628063369
6/93 - Collecting: mt_628585844
7/93 - Collecting: mt_409191960
8/93 - Collecting: mt_017979728
9/93 - Collecting: mt_586363616
10/93 - Collecting: mt_832598327
11/93 - Collecting: mt_365873652
12/93 - Collecting: mt_017420172
13/93 - Collecting: mt_586215868
14/93 - Collecting: mt_970189706
15/93 - Collecting: mt_741937417
16/93 - Collecting: mt_628346289
17/93 - Collecting: mt_253602533
18/93 - Collecting: mt_586174265
19/93 - Collecting: mt_970862103
20/93 - Collecting: mt_970862894
21/93 - Collecting: mt_253097027
22/93 - Collecting: mt_832940983
23/93 - Collecting: mt_017283200
24/93 - Collecting: mt_253097638
25/93 - Collecting: mt_194536046
26/93 - Collecting: mt_741325919
27/93 - Collecting: mt_832940529
28/93 - Collecting: mt_365701856
29/93 - Collecting: mt_628419381
30/93 - Collecting: mt_409658799
31/93 - Collecting:

In [203]:
saved_efl_cup_2021_22 = list(
    EFL_CUP_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved EFL Cup 2021/22 files:", len(saved_efl_cup_2021_22))

Saved EFL Cup 2021/22 files: 93


In [204]:
# Collect 2020/21 FA Cup season

efl_cup_2020_21_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2020_21"]
)

print("Total matches returned:", len(efl_cup_2020_21_matches))

Total matches returned: 90


In [205]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2020_21_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2020_21_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2020_21_matches
)))

Statuses:
Counter({'finished': 90})

Stages:
Counter({None: 83, 'quarter_final': 4, 'semi_final': 2, 'final': 1})

Unique IDs: 90


In [206]:
collect_efl_cup_baseline_data(
    "2020_21",
    efl_cup_2020_21_matches
)

1/90 - Collecting: mt_080205127
2/90 - Collecting: mt_454682996
3/90 - Collecting: mt_616490883
4/90 - Collecting: mt_575758248
5/90 - Collecting: mt_454540787
6/90 - Collecting: mt_080801432
7/90 - Collecting: mt_969697126
8/90 - Collecting: mt_726777244
9/90 - Collecting: mt_291222956
10/90 - Collecting: mt_138111395
11/90 - Collecting: mt_847888434
12/90 - Collecting: mt_610666116
13/90 - Collecting: mt_452444552
14/90 - Collecting: mt_304333065
15/90 - Collecting: mt_969000020
16/90 - Collecting: mt_963940710
17/90 - Collecting: mt_304325678
18/90 - Collecting: mt_085097141
19/90 - Collecting: mt_847862356
20/90 - Collecting: mt_579536827
21/90 - Collecting: mt_452419060
22/90 - Collecting: mt_085097128
23/90 - Collecting: mt_726701496
24/90 - Collecting: mt_610658233
25/90 - Collecting: mt_847862390
26/90 - Collecting: mt_610658244
27/90 - Collecting: mt_138174958
28/90 - Collecting: mt_304325681
29/90 - Collecting: mt_452419076
30/90 - Collecting: mt_726701430
31/90 - Collecting:

In [207]:
saved_efl_cup_2020_21 = list(
    EFL_CUP_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved EFL Cup 2020/21 files:", len(saved_efl_cup_2020_21))

Saved EFL Cup 2020/21 files: 90


In [208]:
# Collect 2024/25 FA Cup season

efl_cup_2024_25_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2024_25"]
)

print("Total matches returned:", len(efl_cup_2024_25_matches))

Total matches returned: 93


In [209]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2024_25_matches
)))

Statuses:
Counter({'finished': 93})

Stages:
Counter({None: 91, 'final': 1, 'semi_final': 1})

Unique IDs: 93


In [210]:
collect_efl_cup_baseline_data(
    "2024_25",
    efl_cup_2024_25_matches
)

1/93 - Collecting: mt_746402015
2/93 - Collecting: mt_401468670
3/93 - Collecting: mt_625649434
4/93 - Collecting: mt_197156557
5/93 - Collecting: mt_258207001
6/93 - Collecting: mt_197648316
7/93 - Collecting: mt_258731531
8/93 - Collecting: mt_197648947
9/93 - Collecting: mt_740516415
10/93 - Collecting: mt_740496537
11/93 - Collecting: mt_740479467
12/93 - Collecting: mt_362684172
13/93 - Collecting: mt_625230949
14/93 - Collecting: mt_974713286
15/93 - Collecting: mt_019145322
16/93 - Collecting: mt_583829418
17/93 - Collecting: mt_836357097
18/93 - Collecting: mt_974779168
19/93 - Collecting: mt_362582672
20/93 - Collecting: mt_974014728
21/93 - Collecting: mt_583623842
22/93 - Collecting: mt_401924512
23/93 - Collecting: mt_974014770
24/93 - Collecting: mt_836256340
25/93 - Collecting: mt_197440830
26/93 - Collecting: mt_019774580
27/93 - Collecting: mt_019774527
28/93 - Collecting: mt_740119635
29/93 - Collecting: mt_625883042
30/93 - Collecting: mt_401997265
31/93 - Collecting:

In [211]:
saved_efl_cup_2024_25 = list(
    EFL_CUP_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved EFL Cup 2024/25 files:", len(saved_efl_cup_2024_25))

Saved EFL Cup 2024/25 files: 93


In [212]:
# Collect 2025/26 FA Cup season

efl_cup_2025_26_matches = get_all_season_matches(
    EFL_CUP_ID,
    EFL_CUP_SEASONS["2025_26"]
)

print("Total matches returned:", len(efl_cup_2025_26_matches))

Total matches returned: 93


In [213]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in efl_cup_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in efl_cup_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in efl_cup_2025_26_matches
)))

Statuses:
Counter({'finished': 93})

Stages:
Counter({None: 84, 'semi_final': 4, 'quarter_final': 4, 'final': 1})

Unique IDs: 93


In [214]:
collect_efl_cup_baseline_data(
    "2025_26",
    efl_cup_2025_26_matches
)

1/93 - Collecting: mt_734246357
2/93 - Collecting: mt_209892338
3/93 - Collecting: mt_209879302
4/93 - Collecting: mt_153763458
5/93 - Collecting: mt_153738138
6/93 - Collecting: mt_894457408
7/93 - Collecting: mt_153308373
8/93 - Collecting: mt_209961985
9/93 - Collecting: mt_732296201
10/93 - Collecting: mt_027578092
11/93 - Collecting: mt_648081659
12/93 - Collecting: mt_980306946
13/93 - Collecting: mt_203139283
14/93 - Collecting: mt_154843161
15/93 - Collecting: mt_731612756
16/93 - Collecting: mt_469295417
17/93 - Collecting: mt_516967538
18/93 - Collecting: mt_980170934
19/93 - Collecting: mt_375865336
20/93 - Collecting: mt_648328661
21/93 - Collecting: mt_516286599
22/93 - Collecting: mt_203653217
23/93 - Collecting: mt_469705995
24/93 - Collecting: mt_203653228
25/93 - Collecting: mt_154094116
26/93 - Collecting: mt_731941779
27/93 - Collecting: mt_892532889
28/93 - Collecting: mt_469709449
29/93 - Collecting: mt_027417050
30/93 - Collecting: mt_980170993
31/93 - Collecting:

In [215]:
saved_efl_cup_2025_26 = list(
    EFL_CUP_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved EFL Cup 2025/26 files:", len(saved_efl_cup_2025_26))

Saved EFL Cup 2025/26 files: 93


In [216]:
# Create code to collect UEFA Europa League seasons

EUROPA_LEAGUE_ID = "comp_7739"

EUROPA_LEAGUE_SEASONS = {
    "2026_27": "sn_9624824",
    "2025_26": "sn_0819325",
    "2024_25": "sn_4596897",
    "2023_24": "sn_008323",
    "2022_23": "sn_706199",
    "2021_22": "sn_481111",
    "2020_21": "sn_522881"
}

In [217]:
# Create a folder for UEFA Europa League matches

EUROPA_LEAGUE_DIR = RAW_MATCH_DIR / "europa_league"

EUROPA_LEAGUE_SEASON_DIRS = {}

for season_name in EUROPA_LEAGUE_SEASONS:
    season_dir = EUROPA_LEAGUE_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    EUROPA_LEAGUE_SEASON_DIRS[season_name] = season_dir

In [218]:
# Create the collector for Europa League seasons

def collect_europa_league_baseline_data(season_name, matches):
    season_dir = EUROPA_LEAGUE_SEASON_DIRS[season_name]
    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(match_id, match_data, season_dir)
        time.sleep(1)

In [ ]:
# Collect 2023/24 Europa League season

europa_2023_24_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2023_24"]
)

print("Total matches returned:", len(europa_2023_24_matches))

Total matches returned: 175


In [220]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2023_24_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2023_24_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2023_24_matches
)))

Statuses:
Counter({'finished': 175})

Stages:
Counter({None: 96, 'round_of_16': 52, 'qualifying': 14, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 175


In [221]:
collect_europa_league_baseline_data(
    "2023_24",
    europa_2023_24_matches
)

1/175 - Collecting: mt_582571997
2/175 - Collecting: mt_971968330
3/175 - Collecting: mt_256290182
4/175 - Collecting: mt_190135878
5/175 - Collecting: mt_835849760
6/175 - Collecting: mt_623614054
7/175 - Collecting: mt_014082598
8/175 - Collecting: mt_749676647
9/175 - Collecting: mt_835787737
10/175 - Collecting: mt_582571936
11/175 - Collecting: mt_971968329
12/175 - Collecting: mt_368434462
13/175 - Collecting: mt_623060029
14/175 - Collecting: mt_749069630
15/175 - Collecting: mt_835675790
16/175 - Collecting: mt_014954528
17/175 - Collecting: mt_971431379
18/175 - Collecting: mt_368248478
19/175 - Collecting: mt_623503044
20/175 - Collecting: mt_407127260
21/175 - Collecting: mt_582392916
22/175 - Collecting: mt_190780836
23/175 - Collecting: mt_835675749
24/175 - Collecting: mt_368248406
25/175 - Collecting: mt_407127259
26/175 - Collecting: mt_256816198
27/175 - Collecting: mt_623503011
28/175 - Collecting: mt_014954570
29/175 - Collecting: mt_749069629
30/175 - Collecting: mt

In [222]:
saved_europa_2023_24 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved Europa League 2023/24 files:", len(saved_europa_2023_24))

Saved Europa League 2023/24 files: 175


In [223]:
# Collect 2022/23 Europa League season

europa_2022_23_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2022_23"]
)

print("Total matches returned:", len(europa_2022_23_matches))

Total matches returned: 175


In [224]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2022_23_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2022_23_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2022_23_matches
)))

Statuses:
Counter({'finished': 175})

Stages:
Counter({None: 96, 'qualifying': 34, 'round_of_16': 32, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 175


In [225]:
collect_europa_league_baseline_data(
    "2022_23",
    europa_2022_23_matches
)

1/175 - Collecting: mt_839332876
2/175 - Collecting: mt_743222403
3/175 - Collecting: mt_978666711
4/175 - Collecting: mt_839444363
5/175 - Collecting: mt_250999560
6/175 - Collecting: mt_199160671
7/175 - Collecting: mt_744759506
8/175 - Collecting: mt_833805066
9/175 - Collecting: mt_366318160
10/175 - Collecting: mt_622693953
11/175 - Collecting: mt_400487816
12/175 - Collecting: mt_400487854
13/175 - Collecting: mt_366318121
14/175 - Collecting: mt_977347442
15/175 - Collecting: mt_400210182
16/175 - Collecting: mt_588938344
17/175 - Collecting: mt_199879769
18/175 - Collecting: mt_255185886
19/175 - Collecting: mt_199879775
20/175 - Collecting: mt_744604004
21/175 - Collecting: mt_833763664
22/175 - Collecting: mt_011591939
23/175 - Collecting: mt_255185870
24/175 - Collecting: mt_366426283
25/175 - Collecting: mt_400210171
26/175 - Collecting: mt_977347421
Request failed: 404
{"error":{"code":"NOT_FOUND","message":"Match statistics not found","status_code":404}}
Saving with missi

In [226]:
saved_europa_2022_23 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved Europa League 2022/23 files:", len(saved_europa_2022_23))

Saved Europa League 2022/23 files: 175


In [227]:
# Collect 2021/22 Europa League season

europa_2021_22_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2021_22"]
)

print("Total matches returned:", len(europa_2021_22_matches))

Total matches returned: 175


In [228]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2021_22_matches
)))

Statuses:
Counter({'finished': 175})

Stages:
Counter({None: 97, 'qualifying': 35, 'round_of_16': 30, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 175


In [229]:
collect_europa_league_baseline_data(
    "2021_22",
    europa_2021_22_matches
)

1/175 - Collecting: mt_742331991
2/175 - Collecting: mt_976746621
3/175 - Collecting: mt_259589970
4/175 - Collecting: mt_193973369
5/175 - Collecting: mt_742402252
6/175 - Collecting: mt_193378500
7/175 - Collecting: mt_742206393
8/175 - Collecting: mt_587739113
9/175 - Collecting: mt_834467953
10/175 - Collecting: mt_360024780
11/175 - Collecting: mt_621150438
12/175 - Collecting: mt_018895225
13/175 - Collecting: mt_405512664
14/175 - Collecting: mt_621181398
15/175 - Collecting: mt_193343005
16/175 - Collecting: mt_018878429
Request failed: 404
{"error":{"code":"NOT_FOUND","message":"Match statistics not found","status_code":404}}
Saving with missing sections: ['statistics']
17/175 - Collecting: mt_742212953
18/175 - Collecting: mt_587767223
19/175 - Collecting: mt_742212994
20/175 - Collecting: mt_405595762
21/175 - Collecting: mt_834424503
22/175 - Collecting: mt_405595774
23/175 - Collecting: mt_834424554
24/175 - Collecting: mt_621181346
25/175 - Collecting: mt_259939666
26/175

In [230]:
saved_europa_2021_22 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved Europa League 2021/22 files:", len(saved_europa_2021_22))

Saved Europa League 2021/22 files: 175


In [ ]:
# Collect 2020/21 Europa League season

europa_2020_21_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2020_21"]
)

print("Total matches returned:", len(europa_2020_21_matches))

Total matches returned: 361


In [232]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2020_21_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2020_21_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2020_21_matches
)))

Statuses:
Counter({'finished': 361})

Stages:
Counter({'qualifying': 157, None: 143, 'round_of_32': 32, 'round_of_16': 16, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 361


In [235]:
collect_europa_league_baseline_data(
    "2020_21",
    europa_2020_21_matches
)

1/361 - Skipping existing: mt_454496751
2/361 - Skipping existing: mt_292108666
3/361 - Skipping existing: mt_727630994
4/361 - Skipping existing: mt_131857005
5/361 - Skipping existing: mt_848796554
6/361 - Skipping existing: mt_303272587
7/361 - Skipping existing: mt_616545832
8/361 - Skipping existing: mt_454161975
9/361 - Skipping existing: mt_575313615
10/361 - Skipping existing: mt_131757407
11/361 - Skipping existing: mt_727030195
12/361 - Skipping existing: mt_848696255
13/361 - Skipping existing: mt_080929747
14/361 - Skipping existing: mt_303114004
15/361 - Skipping existing: mt_616990115
16/361 - Skipping existing: mt_616990810
17/361 - Skipping existing: mt_575449672
18/361 - Skipping existing: mt_575439964
19/361 - Skipping existing: mt_292771955
20/361 - Skipping existing: mt_848007448
21/361 - Skipping existing: mt_080335884
22/361 - Skipping existing: mt_131668393
23/361 - Skipping existing: mt_727556241
24/361 - Skipping existing: mt_969223667
25/361 - Skipping existin

In [236]:
saved_europa_2020_21 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2020_21"].glob("*.json")
)

print("Saved Europa League 2020/21 files:", len(saved_europa_2020_21))

Saved Europa League 2020/21 files: 361


In [237]:
# Collecting 2024/25 Europa League season

europa_2024_25_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2024_25"]
)

print("Total matches returned:", len(europa_2024_25_matches))

Total matches returned: 269


In [238]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2024_25_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2024_25_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2024_25_matches
)))

Statuses:
Counter({'finished': 269})

Stages:
Counter({None: 144, 'round_of_16': 56, 'qualifying': 56, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 269


In [239]:
collect_europa_league_baseline_data(
    "2024_25",
    europa_2024_25_matches
)

1/269 - Collecting: mt_973106364
2/269 - Collecting: mt_251639197
3/269 - Collecting: mt_746366930
4/269 - Collecting: mt_402622760
5/269 - Collecting: mt_015255428
6/269 - Collecting: mt_837977590
7/269 - Collecting: mt_364744878
8/269 - Collecting: mt_620400344
9/269 - Collecting: mt_589199216
10/269 - Collecting: mt_589199269
11/269 - Collecting: mt_973833104
12/269 - Collecting: mt_251011637
13/269 - Collecting: mt_198588040
14/269 - Collecting: mt_620403369
15/269 - Collecting: mt_589192258
16/269 - Collecting: mt_746369966
17/269 - Collecting: mt_198580078
18/269 - Collecting: mt_620403985
19/269 - Collecting: mt_973831207
20/269 - Collecting: mt_589192297
21/269 - Collecting: mt_402627710
22/269 - Collecting: mt_015254374
23/269 - Collecting: mt_746369977
24/269 - Collecting: mt_973831130
25/269 - Collecting: mt_015254498
26/269 - Collecting: mt_402627747
27/269 - Collecting: mt_251016623
Request failed: 404
{"error":{"code":"NOT_FOUND","message":"Odds not found for this match",

In [240]:
saved_europa_2024_25 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved Europa League 2024/25 files:", len(saved_europa_2024_25))

Saved Europa League 2024/25 files: 269


In [241]:
# Collect 2025/26 Europa League season

europa_2025_26_matches = get_all_season_matches(
    EUROPA_LEAGUE_ID,
    EUROPA_LEAGUE_SEASONS["2025_26"]
)

print("Total matches returned:", len(europa_2025_26_matches))

Total matches returned: 271


In [242]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in europa_2025_26_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in europa_2025_26_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in europa_2025_26_matches
)))

Statuses:
Counter({'finished': 271})

Stages:
Counter({None: 144, 'qualifying': 58, 'round_of_16': 56, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 271


In [243]:
collect_europa_league_baseline_data(
    "2025_26",
    europa_2025_26_matches
)

1/271 - Collecting: mt_642459643
2/271 - Collecting: mt_460618466
3/271 - Collecting: mt_021293021
4/271 - Collecting: mt_518134517
5/271 - Collecting: mt_987842980
6/271 - Collecting: mt_205087252
7/271 - Collecting: mt_159576198
8/271 - Collecting: mt_893960830
9/271 - Collecting: mt_734305740
10/271 - Collecting: mt_376721368
11/271 - Collecting: mt_642459624
12/271 - Collecting: mt_460618400
13/271 - Collecting: mt_021293018
14/271 - Collecting: mt_518132932
15/271 - Collecting: mt_205086185
16/271 - Collecting: mt_159570873
17/271 - Collecting: mt_376728414
18/271 - Collecting: mt_021294593
19/271 - Collecting: mt_987841348
20/271 - Collecting: mt_893965708
21/271 - Collecting: mt_734309601
22/271 - Collecting: mt_460617288
23/271 - Collecting: mt_021294534
24/271 - Collecting: mt_159570864
25/271 - Collecting: mt_376728482
26/271 - Collecting: mt_642453095
27/271 - Collecting: mt_518132941
28/271 - Collecting: mt_987841327
29/271 - Collecting: mt_893965757
30/271 - Collecting: mt

In [244]:
saved_europa_2025_26 = list(
    EUROPA_LEAGUE_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved Europa League 2025/26 files:", len(saved_europa_2025_26))

Saved Europa League 2025/26 files: 271


In [ ]:
# Set Up UEFA Conference League seasons

CONFERENCE_LEAGUE_ID = "comp_408698"

CONFERENCE_LEAGUE_SEASONS = {
    "2026_27": "sn_7250309",
    "2025_26": "sn_1397633",
    "2024_25": "sn_8429027",
    "2023_24": "sn_343450",
    "2022_23": "sn_092013",
    "2021_22": "sn_033592"
}

In [246]:
# Create folder for UEFA Conference League matches

CONFERENCE_LEAGUE_DIR = RAW_MATCH_DIR / "conference_league"

CONFERENCE_LEAGUE_SEASON_DIRS = {}

for season_name in CONFERENCE_LEAGUE_SEASONS:
    season_dir = CONFERENCE_LEAGUE_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    CONFERENCE_LEAGUE_SEASON_DIRS[season_name] = season_dir

In [247]:
# Create a collector for UEFA Conference League seasons

def collect_conference_league_baseline_data(season_name, matches):
    season_dir = CONFERENCE_LEAGUE_SEASON_DIRS[season_name]
    total_matches = len(matches)

    for i, match in enumerate(matches, start=1):
        match_id = match["id"]
        file_path = season_dir / f"{match_id}.json"

        if file_path.exists():
            print(f"{i}/{total_matches} - Skipping existing: {match_id}")
            continue

        print(f"{i}/{total_matches} - Collecting: {match_id}")

        match_data = get_baseline_match_data(match_id)

        if match_data is None:
            print("Skipped:", match_id)
            continue

        save_match_data(match_id, match_data, season_dir)

        time.sleep(1)

In [248]:
# Collect 2021/22 Conference League season

conference_2021_22_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2021_22"]
)

print("Total matches returned:", len(conference_2021_22_matches))

Total matches returned: 423


In [249]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2021_22_matches))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2021_22_matches))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2021_22_matches
)))

Statuses:
Counter({'finished': 423})

Stages:
Counter({'qualifying': 282, None: 112, 'round_of_16': 16, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 423


In [250]:
conference_2021_22_filtered = [
    match
    for match in conference_2021_22_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2021_22_matches))
print("Matches after filtering:", len(conference_2021_22_filtered))

Matches before filtering: 423
Matches after filtering: 141


In [251]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2021_22_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2021_22_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2021_22_filtered
)))

Statuses:
Counter({'finished': 141})

Stages:
Counter({None: 112, 'round_of_16': 16, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 141


In [252]:
collect_conference_league_baseline_data(
    "2021_22",
    conference_2021_22_filtered
)

1/141 - Collecting: mt_834992598
2/141 - Collecting: mt_018198735
3/141 - Collecting: mt_587837643
4/141 - Collecting: mt_193973465
5/141 - Collecting: mt_742402154
6/141 - Collecting: mt_976643341
7/141 - Collecting: mt_360024413
8/141 - Collecting: mt_405512281
9/141 - Collecting: mt_018895533
10/141 - Collecting: mt_742206740
11/141 - Collecting: mt_621150090
12/141 - Collecting: mt_976643328
13/141 - Collecting: mt_587739942
14/141 - Collecting: mt_587767235
15/141 - Collecting: mt_742212909
16/141 - Collecting: mt_018878430
17/141 - Collecting: mt_587767249
18/141 - Collecting: mt_018878457
19/141 - Collecting: mt_976606143
20/141 - Collecting: mt_405595719
21/141 - Collecting: mt_976606124
22/141 - Collecting: mt_621181302
23/141 - Collecting: mt_360050826
24/141 - Collecting: mt_621181351
25/141 - Collecting: mt_259939677
26/141 - Collecting: mt_405595725
27/141 - Collecting: mt_259939688
28/141 - Collecting: mt_193343076
29/141 - Collecting: mt_834424569
30/141 - Collecting: mt

In [253]:
saved_conference_2021_22 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2021_22"].glob("*.json")
)

print("Saved Conference League 2021/22 files:", len(saved_conference_2021_22))

Saved Conference League 2021/22 files: 141


In [254]:
# Collect 2022/23 Conference League season

conference_2022_23_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2022_23"]
)

print("Total matches returned:", len(conference_2022_23_matches))

Total matches returned: 415


In [255]:
conference_2022_23_filtered = [
    match
    for match in conference_2022_23_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2022_23_matches))
print("Matches after filtering:", len(conference_2022_23_filtered))

Matches before filtering: 415
Matches after filtering: 141


In [256]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2022_23_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2022_23_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2022_23_filtered
)))

Statuses:
Counter({'finished': 141})

Stages:
Counter({None: 96, 'round_of_16': 32, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 141


In [257]:
collect_conference_league_baseline_data(
    "2022_23",
    conference_2022_23_filtered
)

1/141 - Collecting: mt_406005964
2/141 - Collecting: mt_581777843
3/141 - Collecting: mt_978666722
4/141 - Collecting: mt_250999576
5/141 - Collecting: mt_195333965
6/141 - Collecting: mt_011034915
7/141 - Collecting: mt_199160795
8/141 - Collecting: mt_977921472
9/141 - Collecting: mt_255276856
10/141 - Collecting: mt_833805634
11/141 - Collecting: mt_366318265
12/141 - Collecting: mt_622693516
13/141 - Collecting: mt_744759044
14/141 - Collecting: mt_400210212
15/141 - Collecting: mt_011591599
16/141 - Collecting: mt_977347341
17/141 - Collecting: mt_255185180
18/141 - Collecting: mt_366426445
19/141 - Collecting: mt_622052056
20/141 - Collecting: mt_588938934
21/141 - Collecting: mt_366426307
22/141 - Collecting: mt_622052612
23/141 - Collecting: mt_833763762
24/141 - Collecting: mt_366426413
25/141 - Collecting: mt_011591533
26/141 - Collecting: mt_744604602
27/141 - Collecting: mt_400210281
28/141 - Collecting: mt_588938942
Request failed: 404
{"error":{"code":"NOT_FOUND","message

In [258]:
saved_conference_2022_23 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2022_23"].glob("*.json")
)

print("Saved Conference League 2022/23 files:", len(saved_conference_2022_23))

Saved Conference League 2022/23 files: 141


In [259]:
# Collect 2023/24 Conference League season

conference_2023_24_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2023_24"]
)

print("Total matches returned:", len(conference_2023_24_matches))

Total matches returned: 417


In [260]:
conference_2023_24_filtered = [
    match
    for match in conference_2023_24_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2023_24_matches))
print("Matches after filtering:", len(conference_2023_24_filtered))

Matches before filtering: 417
Matches after filtering: 185


In [261]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2023_24_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2023_24_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2023_24_filtered
)))

Statuses:
Counter({'finished': 185})

Stages:
Counter({None: 96, 'round_of_16': 76, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 185


In [263]:
collect_conference_league_baseline_data(
    "2023_24",
    conference_2023_24_filtered
)

1/185 - Skipping existing: mt_835849894
2/185 - Skipping existing: mt_623614626
3/185 - Skipping existing: mt_368307375
4/185 - Skipping existing: mt_407456402
5/185 - Collecting: mt_014082019
6/185 - Collecting: mt_582571584
7/185 - Collecting: mt_971968971
8/185 - Collecting: mt_256290250
9/185 - Collecting: mt_190135199
10/185 - Collecting: mt_835787858
11/185 - Collecting: mt_256121269
12/185 - Collecting: mt_190818163
13/185 - Collecting: mt_014505044
14/185 - Collecting: mt_835688905
15/185 - Collecting: mt_368233717
16/185 - Collecting: mt_623566492
17/185 - Collecting: mt_407144685
18/185 - Collecting: mt_014900237
19/185 - Collecting: mt_582355125
20/185 - Collecting: mt_971499813
21/185 - Collecting: mt_256822068
22/185 - Collecting: mt_835675895
23/185 - Collecting: mt_407127465
24/185 - Collecting: mt_014954027
25/185 - Collecting: mt_835675843
26/185 - Collecting: mt_256816201
27/185 - Collecting: mt_623503642
28/185 - Collecting: mt_749069749
29/185 - Collecting: mt_97143

In [264]:
saved_conference_2023_24 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2023_24"].glob("*.json")
)

print("Saved Conference League 2023/24 files:", len(saved_conference_2023_24))

Saved Conference League 2023/24 files: 185


In [265]:
# Collect 2024/25 Europa League season

conference_2024_25_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2024_25"]
)

print("Total matches returned:", len(conference_2024_25_matches))

Total matches returned: 407


In [266]:
conference_2024_25_filtered = [
    match
    for match in conference_2024_25_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2024_25_matches))
print("Matches after filtering:", len(conference_2024_25_filtered))

Matches before filtering: 407
Matches after filtering: 201


In [267]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2024_25_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2024_25_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2024_25_filtered
)))

Statuses:
Counter({'finished': 201})

Stages:
Counter({None: 108, 'round_of_16': 80, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 201


In [270]:
collect_conference_league_baseline_data(
    "2024_25",
    conference_2024_25_filtered
)

1/201 - Skipping existing: mt_973106302
2/201 - Skipping existing: mt_837977734
3/201 - Skipping existing: mt_015255573
4/201 - Skipping existing: mt_364744465
5/201 - Skipping existing: mt_402622291
6/201 - Skipping existing: mt_973833497
7/201 - Skipping existing: mt_251011829
8/201 - Skipping existing: mt_015255589
9/201 - Skipping existing: mt_364744234
10/201 - Skipping existing: mt_589199351
11/201 - Skipping existing: mt_198588714
12/201 - Skipping existing: mt_746366067
13/201 - Skipping existing: mt_402622252
14/201 - Skipping existing: mt_589194765
15/201 - Skipping existing: mt_251017938
16/201 - Skipping existing: mt_364741056
17/201 - Skipping existing: mt_198586485
18/201 - Skipping existing: mt_746365219
19/201 - Skipping existing: mt_251017327
20/201 - Skipping existing: mt_837970283
21/201 - Skipping existing: mt_837970274
22/201 - Skipping existing: mt_837970429
23/201 - Skipping existing: mt_198586410
24/201 - Skipping existing: mt_364741530
25/201 - Skipping existin

In [271]:
saved_conference_2024_25 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2024_25"].glob("*.json")
)

print("Saved Conference League 2024/25 files:", len(saved_conference_2024_25))

Saved Conference League 2024/25 files: 201


In [272]:
# Collect 2025/26 Conference League season

conference_2025_26_matches = get_all_season_matches(
    CONFERENCE_LEAGUE_ID,
    CONFERENCE_LEAGUE_SEASONS["2025_26"]
)

print("Total matches returned:", len(conference_2025_26_matches))

Total matches returned: 411


In [273]:
conference_2025_26_filtered = [
    match
    for match in conference_2025_26_matches
    if match.get("stage_name") != "qualifying"
]

print("Matches before filtering:", len(conference_2025_26_matches))
print("Matches after filtering:", len(conference_2025_26_filtered))

Matches before filtering: 411
Matches after filtering: 201


In [275]:
from collections import Counter

print("Statuses:")
print(Counter(match["status"] for match in conference_2025_26_filtered))

print("\nStages:")
print(Counter(match.get("stage_name") for match in conference_2025_26_filtered))

print("\nUnique IDs:", len(set(
    match["id"] for match in conference_2025_26_filtered
)))

Statuses:
Counter({'finished': 201})

Stages:
Counter({None: 108, 'round_of_16': 80, 'quarter_final': 8, 'semi_final': 4, 'final': 1})

Unique IDs: 201


In [276]:
collect_conference_league_baseline_data(
    "2025_26",
    conference_2025_26_filtered
)

1/201 - Collecting: mt_734305701
2/201 - Collecting: mt_893960808
3/201 - Collecting: mt_376721314
4/201 - Collecting: mt_642459695
5/201 - Collecting: mt_460618488
6/201 - Collecting: mt_021293034
7/201 - Collecting: mt_518134541
8/201 - Collecting: mt_987842927
9/201 - Collecting: mt_205087279
10/201 - Collecting: mt_159576164
11/201 - Collecting: mt_893960857
12/201 - Collecting: mt_376721382
13/201 - Collecting: mt_734305797
14/201 - Collecting: mt_159576356
15/201 - Collecting: mt_376721076
16/201 - Collecting: mt_642459141
17/201 - Collecting: mt_460618569
18/201 - Collecting: mt_734305239
19/201 - Collecting: mt_893960499
20/201 - Collecting: mt_021293810
21/201 - Collecting: mt_518134789
22/201 - Collecting: mt_205087957
23/201 - Collecting: mt_159576390
24/201 - Collecting: mt_642459128
25/201 - Collecting: mt_460618554
26/201 - Collecting: mt_987842674
27/201 - Collecting: mt_734305243
28/201 - Collecting: mt_893960433
29/201 - Collecting: mt_376721060
30/201 - Collecting: mt

In [277]:
saved_conference_2025_26 = list(
    CONFERENCE_LEAGUE_SEASON_DIRS["2025_26"].glob("*.json")
)

print("Saved Conference League 2025/26 files:", len(saved_conference_2025_26))

Saved Conference League 2025/26 files: 201


In [278]:
PREMIER_LEAGUE_DIR = RAW_MATCH_DIR / "premier_league"

SEASON_DIRS = {}

for season_name in SEASONS:
    season_dir = PREMIER_LEAGUE_DIR / season_name
    season_dir.mkdir(parents=True, exist_ok=True)
    SEASON_DIRS[season_name] = season_dir

In [279]:
for season_name, season_dir in SEASON_DIRS.items():
    print(season_name, "->", season_dir)

2026_27 -> ..\data\raw\matches\premier_league\2026_27
2025_26 -> ..\data\raw\matches\premier_league\2025_26
2024_25 -> ..\data\raw\matches\premier_league\2024_25
2023_24 -> ..\data\raw\matches\premier_league\2023_24
2022_23 -> ..\data\raw\matches\premier_league\2022_23
2021_22 -> ..\data\raw\matches\premier_league\2021_22
2020_21 -> ..\data\raw\matches\premier_league\2020_21


In [298]:
# Updating get_stat for the matches with no available statistics

def get_stat(stats, stat_name, side):
    if stats is None:
        return None

    stats = stats.get("data")

    if stats is None:
        return None
    
    overview = stats.get("overview")

    if overview is None:
        return None

    stat = overview.get(stat_name)

    if stat is None or stat.get("all") is None:
        return None

    return stat["all"].get(side)

In [291]:
# Update historical match extractor so every processed row includes both season and competition 

def extract_historical_match(match_data, season, competition, match_id):
    details = match_data["details"]["data"]
    stats = match_data.get("statistics")

    return {
        "match_id": match_id,
        "date": details["utc_date"],
        "season": season,
        "competition": competition,
        "matchday": details.get("matchday"),

        "home_team": details["home_team"]["name"],
        "away_team": details["away_team"]["name"],

        "home_goals": details["score"]["home"],
        "away_goals": details["score"]["away"],

        "home_xg": get_stat(stats, "expected_goals", "home"),
        "away_xg": get_stat(stats, "expected_goals", "away"),

        "home_shots": get_stat(stats, "total_shots", "home"),
        "away_shots": get_stat(stats, "total_shots", "away"),

        "home_shots_on_target": get_stat(stats, "shots_on_target", "home"),
        "away_shots_on_target": get_stat(stats, "shots_on_target", "away"),

        "home_possession": get_stat(stats, "ball_possession", "home"),
        "away_possession": get_stat(stats, "ball_possession", "away"),

        "home_passes": get_stat(stats, "passes", "home"),
        "away_passes": get_stat(stats, "passes", "away"),

        "home_corners": get_stat(stats, "corner_kicks", "home"),
        "away_corners": get_stat(stats, "corner_kicks", "away"),

        "winner": details["score"].get("winner")
    }

In [282]:
# Create a map telling Python which folder belongs to which competition.

COMPETITION_DIRS = {
    "premier_league": RAW_MATCH_DIR / "premier_league",
    "championship": RAW_MATCH_DIR / "championship",
    "champions_league": RAW_MATCH_DIR / "champions_league",
    "fa_cup": RAW_MATCH_DIR / "fa_cup",
    "efl_cup": RAW_MATCH_DIR / "efl_cup",
    "europa_league": RAW_MATCH_DIR / "europa_league",
    "conference_league": RAW_MATCH_DIR / "conference_league"
}

In [283]:
for competition, directory in COMPETITION_DIRS.items():
    print(competition, "->", directory.exists(), "->", directory)

premier_league -> True -> ..\data\raw\matches\premier_league
championship -> True -> ..\data\raw\matches\championship
champions_league -> True -> ..\data\raw\matches\champions_league
fa_cup -> True -> ..\data\raw\matches\fa_cup
efl_cup -> True -> ..\data\raw\matches\efl_cup
europa_league -> True -> ..\data\raw\matches\europa_league
conference_league -> True -> ..\data\raw\matches\conference_league


In [ ]:
# Separate current season (2026/27) from historical seasons

COMPLETED_SEASONS = {
    "premier_league": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "championship": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "champions_league": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "fa_cup": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "efl_cup": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "europa_league": ["2020_21", "2021_22", "2022_23", "2023_24", "2024_25", "2025_26"],
    "conference_league": ["2021_22", "2022_23", "2023_24", "2024_25", "2025_26"]
}

for competition, seasons in COMPLETED_SEASONS.items():
    print(competition, "->", len(seasons), "seasons")

premier_league -> 6 seasons
championship -> 6 seasons
champions_league -> 6 seasons
fa_cup -> 6 seasons
efl_cup -> 6 seasons
europa_league -> 6 seasons
conference_league -> 5 seasons


In [300]:
# Build loop that reads every historical JSON file and sends it through the extractor

all_historical_matches = []

for competition, seasons in COMPLETED_SEASONS.items():
    competition_dir = COMPETITION_DIRS[competition]

    for season in seasons:
        season_dir = competition_dir / season

        for file_path in season_dir.glob("*.json"):
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    match_data = json.load(f)

                match_id = file_path.stem

                match_row = extract_historical_match(
                    match_data,
                    season,
                    competition,
                    match_id
                )

                all_historical_matches.append(match_row)

            except Exception as e:
                print("Error processing:", file_path)
                print("Error:", e)

print("Total completed historical matches processed:", len(all_historical_matches))

Total completed historical matches processed: 10653


In [301]:
# Turn the list into a DataFrame

historical_df = pd.DataFrame(all_historical_matches)

print(historical_df.shape)
print()
print(historical_df.head())
print()
print(historical_df["competition"].value_counts())

(10653, 22)

       match_id                      date   season     competition  matchday  \
0  mt_085997104  2020-12-16T18:00:00.000Z  2020_21  premier_league      13.0   
1  mt_085997110  2020-10-04T15:30:00.000Z  2020_21  premier_league       4.0   
2  mt_085997138  2020-09-20T11:00:00.000Z  2020_21  premier_league       2.0   
3  mt_085997147  2020-12-27T12:00:00.000Z  2020_21  premier_league      15.0   
4  mt_085997152  2020-09-12T16:30:00.000Z  2020_21  premier_league       1.0   

           home_team          away_team  home_goals  away_goals  home_xg  ...  \
0     Leicester City            Everton           0           2      NaN  ...   
1  Manchester United  Tottenham Hotspur           1           6      NaN  ...   
2        Southampton  Tottenham Hotspur           2           5      NaN  ...   
3       Leeds United            Burnley           1           0      NaN  ...   
4          Liverpool       Leeds United           4           3      NaN  ...   

   away_shots  home

In [304]:
# Check which columns are complete and which ones are sparse before saving into CSV

missing_summary = historical_df.isna().sum().sort_values(ascending=False)

print(missing_summary)

away_xg                 5435
home_xg                 5435
home_passes             1232
away_passes             1232
away_possession         1147
home_possession         1147
away_shots_on_target     966
home_shots_on_target     966
home_shots               966
away_shots               966
away_corners             862
home_corners             862
matchday                 444
match_id                   0
home_team                  0
competition                0
season                     0
date                       0
home_goals                 0
away_team                  0
away_goals                 0
winner                     0
dtype: int64


In [305]:
historical_df.to_csv(
    "../data/processed/historical_matches.csv",
    index=False
)

print("Saved:", historical_df.shape)

Saved: (10653, 22)


In [306]:
import json
from pathlib import Path

sample_file = next(
    (
        Path("../data/raw/matches/premier_league/2024_25")
    ).glob("*.json")
)

with open(
    sample_file,
    "r",
    encoding="utf-8"
) as f:
    sample_match = json.load(f)

print("Match file:", sample_file.name)
print()
print(
    json.dumps(
        sample_match.get("odds"),
        indent=2
    )[:5000]
)

Match file: mt_013233107.json

{
  "data": {
    "match_id": "mt_013233107",
    "bookmakers": [
      {
        "bookmaker": "Bet365",
        "markets": {
          "match_odds": {
            "home": {
              "opening": null,
              "last_seen": "4.100"
            },
            "draw": {
              "opening": null,
              "last_seen": "3.400"
            },
            "away": {
              "opening": null,
              "last_seen": "1.950"
            }
          },
          "btts": {
            "yes": {
              "opening": null,
              "last_seen": "1.910"
            },
            "no": {
              "opening": null,
              "last_seen": "1.910"
            }
          },
          "total_goals": {
            "0.5": {
              "over": {
                "opening": null,
                "last_seen": "1.060"
              },
              "under": {
                "opening": null,
                "last_seen": "10.000"
      

In [307]:
def extract_bet365_match_odds(match_data, match_id):

    odds_data = match_data.get("odds")

    if odds_data is None:
        return None

    odds_data = odds_data.get("data")

    if odds_data is None:
        return None

    bookmakers = odds_data.get("bookmakers", [])

    bet365 = None

    for bookmaker in bookmakers:
        if bookmaker.get("bookmaker") == "Bet365":
            bet365 = bookmaker
            break

    if bet365 is None:
        return None

    match_odds = (
        bet365
        .get("markets", {})
        .get("match_odds")
    )

    if match_odds is None:
        return None

    try:
        home_odds = float(
            match_odds["home"]["last_seen"]
        )

        draw_odds = float(
            match_odds["draw"]["last_seen"]
        )

        away_odds = float(
            match_odds["away"]["last_seen"]
        )

    except (TypeError, ValueError, KeyError):
        return None

    return {
        "match_id": match_id,
        "bet365_home_odds": home_odds,
        "bet365_draw_odds": draw_odds,
        "bet365_away_odds": away_odds
    }

In [308]:
sample_odds = extract_bet365_match_odds(
    sample_match,
    sample_file.stem
)

print(sample_odds)

{'match_id': 'mt_013233107', 'bet365_home_odds': 4.1, 'bet365_draw_odds': 3.4, 'bet365_away_odds': 1.95}


In [309]:
# See how much Bet365 coverage we have across 6 completed PL seasons

bet365_odds_rows = []

for season in COMPLETED_SEASONS["premier_league"]:

    season_dir = (
        COMPETITION_DIRS["premier_league"]
        / season
    )

    season_total = 0
    season_with_odds = 0

    for file_path in season_dir.glob("*.json"):

        season_total += 1

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:
            match_data = json.load(f)

        odds_row = extract_bet365_match_odds(
            match_data,
            file_path.stem
        )

        if odds_row is not None:

            odds_row["season"] = season

            bet365_odds_rows.append(
                odds_row
            )

            season_with_odds += 1

    print(
        f"{season}: "
        f"{season_with_odds}/{season_total} "
        f"matches with Bet365 odds"
    )

print()
print(
    "Total Bet365 matches:",
    len(bet365_odds_rows)
)

2020_21: 380/380 matches with Bet365 odds
2021_22: 380/380 matches with Bet365 odds
2022_23: 380/380 matches with Bet365 odds
2023_24: 380/380 matches with Bet365 odds
2024_25: 380/380 matches with Bet365 odds
2025_26: 380/380 matches with Bet365 odds

Total Bet365 matches: 2280


In [310]:
# Turn these 2,280 rows into a DataFrame and calculate fair market probabilities by removing Bet365 margin

bet365_df = pd.DataFrame(
    bet365_odds_rows
)

# Raw implied probabilities
bet365_df["raw_prob_H"] = (
    1 / bet365_df["bet365_home_odds"]
)

bet365_df["raw_prob_D"] = (
    1 / bet365_df["bet365_draw_odds"]
)

bet365_df["raw_prob_A"] = (
    1 / bet365_df["bet365_away_odds"]
)

# Bookmaker overround
bet365_df["overround"] = (
    bet365_df["raw_prob_H"]
    + bet365_df["raw_prob_D"]
    + bet365_df["raw_prob_A"]
)

# Normalize to remove bookmaker margin
bet365_df["fair_prob_H"] = (
    bet365_df["raw_prob_H"]
    / bet365_df["overround"]
)

bet365_df["fair_prob_D"] = (
    bet365_df["raw_prob_D"]
    / bet365_df["overround"]
)

bet365_df["fair_prob_A"] = (
    bet365_df["raw_prob_A"]
    / bet365_df["overround"]
)

print(bet365_df.shape)

print()

print(
    bet365_df[
        [
            "bet365_home_odds",
            "bet365_draw_odds",
            "bet365_away_odds",
            "overround",
            "fair_prob_H",
            "fair_prob_D",
            "fair_prob_A"
        ]
    ].head()
)

print()

print(
    "Average overround:",
    bet365_df["overround"].mean()
)

print(
    "Average fair probability sum:",
    (
        bet365_df["fair_prob_H"]
        + bet365_df["fair_prob_D"]
        + bet365_df["fair_prob_A"]
    ).mean()
)

(2280, 12)

   bet365_home_odds  bet365_draw_odds  bet365_away_odds  overround  \
0              2.10               3.4               3.5   1.056022   
1              2.00               3.7               3.6   1.048048   
2              3.30               3.4               2.2   1.051693   
3              1.70               4.2               4.5   1.048553   
4              1.25               6.0              11.0   1.057576   

   fair_prob_H  fair_prob_D  fair_prob_A  
0     0.450928     0.278515     0.270557  
1     0.477077     0.257880     0.265043  
2     0.288136     0.279661     0.432203  
3     0.560997     0.227070     0.211932  
4     0.756447     0.157593     0.085960  

Average overround: 1.0545628863489702
Average fair probability sum: 1.0


In [311]:
# Save Bet365 data
bet365_df.to_csv(
    "../data/processed/bet365_pl_odds.csv",
    index=False
)

print(
    "Saved:",
    bet365_df.shape
)

Saved: (2280, 12)


In [322]:
# ---------------------------------------------------------
# PROCESS ONE RAW MATCH FILE
# ---------------------------------------------------------

def process_match_file(file_path):

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            match_data = json.load(f)

    except Exception as e:

        print(
            f"Could not read {file_path}: {e}"
        )

        return None


    # -----------------------------------------------------
    # DETAILS
    # -----------------------------------------------------

    details = match_data.get(
        "details"
    )

    if details is None:
        return None

    if isinstance(details, dict):

        details = details.get(
            "data",
            details
        )

    if not isinstance(details, dict):
        return None


    # Only completed matches
    if details.get("status") != "finished":
        return None


    # -----------------------------------------------------
    # STATISTICS
    # -----------------------------------------------------

    statistics = match_data.get(
        "statistics"
    )


    # -----------------------------------------------------
    # TEAM INFORMATION
    # -----------------------------------------------------

    home_team_data = details.get(
        "home_team",
        {}
    )

    away_team_data = details.get(
        "away_team",
        {}
    )


    home_team = (
        home_team_data.get("name")
        if isinstance(home_team_data, dict)
        else None
    )

    away_team = (
        away_team_data.get("name")
        if isinstance(away_team_data, dict)
        else None
    )


    # -----------------------------------------------------
    # SCORE
    # -----------------------------------------------------

    score = details.get(
        "score",
        {}
    )

    if not isinstance(score, dict):
        score = {}


    home_goals = score.get(
        "home"
    )

    away_goals = score.get(
        "away"
    )


    # -----------------------------------------------------
    # WINNER
    # -----------------------------------------------------

    winner = None

    if (
        home_goals is not None
        and away_goals is not None
    ):

        if home_goals > away_goals:

            winner = "H"

        elif away_goals > home_goals:

            winner = "A"

        else:

            winner = "D"


    # -----------------------------------------------------
    # FOLDER INFORMATION
    # -----------------------------------------------------

    season = file_path.parent.name

    competition = (
        file_path
        .parent
        .parent
        .name
    )


    # -----------------------------------------------------
    # BUILD ROW
    # -----------------------------------------------------

    row = {

        "match_id":
            details.get("id"),

        "date":
            details.get("utc_date"),

        "season":
            season,

        "competition":
            competition,

        "matchday":
            details.get("matchday"),

        "home_team":
            home_team,

        "away_team":
            away_team,

        "home_goals":
            home_goals,

        "away_goals":
            away_goals,

        "home_xg":
            get_stat(
                statistics,
                "home",
                "expected_goals"
            ),

        "away_xg":
            get_stat(
                statistics,
                "away",
                "expected_goals"
            ),

        "home_shots":
            get_stat(
                statistics,
                "home",
                "shots"
            ),

        "away_shots":
            get_stat(
                statistics,
                "away",
                "shots"
            ),

        "home_shots_on_target":
            get_stat(
                statistics,
                "home",
                "shots_on_target"
            ),

        "away_shots_on_target":
            get_stat(
                statistics,
                "away",
                "shots_on_target"
            ),

        "home_possession":
            get_stat(
                statistics,
                "home",
                "ball_possession"
            ),

        "away_possession":
            get_stat(
                statistics,
                "away",
                "ball_possession"
            ),

        "home_passes":
            get_stat(
                statistics,
                "home",
                "passes"
            ),

        "away_passes":
            get_stat(
                statistics,
                "away",
                "passes"
            ),

        "home_corners":
            get_stat(
                statistics,
                "home",
                "corners"
            ),

        "away_corners":
            get_stat(
                statistics,
                "away",
                "corners"
            ),

        "winner":
            winner
    }


    return row

In [323]:
# ---------------------------------------------------------
# CHECK 2026/27 PROCESSING
# ---------------------------------------------------------

check_2026 = historical_df[
    (
        historical_df["competition"] == "premier_league"
    )
    &
    (
        historical_df["season"] == "2026_27"
    )
][
    [
        "date",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "winner"
    ]
].sort_values("date")


print(
    check_2026
)

print()

print(
    "Missing dates:",
    check_2026["date"].isna().sum()
)

print(
    "Missing home goals:",
    check_2026["home_goals"].isna().sum()
)

print(
    "Missing away goals:",
    check_2026["away_goals"].isna().sum()
)

      date               home_team               away_team home_goals  \
850    NaT                  Fulham                 Chelsea       None   
851    NaT               Liverpool       Nottingham Forest       None   
852    NaT               Hull City       Manchester United       None   
1328   NaT           Coventry City               Hull City       None   
4259   NaT             Bournemouth                 Everton       None   
5304   NaT                 Everton          Crystal Palace       None   
5305   NaT             Aston Villa                 Arsenal       None   
5306   NaT              Sunderland                  Fulham       None   
5363   NaT                 Arsenal           Coventry City       None   
5364   NaT         Manchester City             Bournemouth       None   
5365   NaT       Manchester United            Ipswich Town       None   
7393   NaT  Brighton & Hove Albion             Aston Villa       None   
7394   NaT       Tottenham Hotspur        Newcastle

In [324]:
import inspect

print(
    inspect.getsource(
        process_match_file
    )
)

def process_match_file(file_path):

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            match_data = json.load(f)

    except Exception as e:

        print(
            f"Could not read {file_path}: {e}"
        )

        return None


    # -----------------------------------------------------
    # DETAILS
    # -----------------------------------------------------

    details = match_data.get(
        "details"
    )

    if details is None:
        return None

    if isinstance(details, dict):

        details = details.get(
            "data",
            details
        )

    if not isinstance(details, dict):
        return None


    # Only completed matches
    if details.get("status") != "finished":
        return None


    # -----------------------------------------------------
    # STATISTICS
    # -----------------------------------------------------

    statistics = match_data.get(
        "

In [326]:
# ---------------------------------------------------------
# INSPECT ONE 2026/27 RAW MATCH
# ---------------------------------------------------------

import json
from pathlib import Path
from pprint import pprint


season_dir = Path(
    "../data/raw/matches/premier_league/2026_27"
)


match_files = list(
    season_dir.glob("*.json")
)


print(
    "2026/27 raw files:",
    len(match_files)
)


with open(
    match_files[0],
    "r",
    encoding="utf-8"
) as f:

    sample_match = json.load(f)


print("\nTOP-LEVEL KEYS:")
print(
    sample_match.keys()
)


print("\nDETAILS:")
pprint(
    sample_match.get("details")
)

2026/27 raw files: 20

TOP-LEVEL KEYS:
dict_keys(['details', 'statistics', 'lineups', 'player_stats', 'odds'])

DETAILS:
{'data': {'away_team': {'id': 'tm_8531', 'name': 'Chelsea'},
          'competition_id': 'comp_3039',
          'competition_name': 'Premier League',
          'group_label': None,
          'home_team': {'id': 'tm_5290', 'name': 'Fulham'},
          'id': 'mt_022587821',
          'is_neutral': False,
          'live_odds_available': False,
          'matchday': 1,
          'odds_available': True,
          'referee': {'id': 'ref_29658200', 'name': 'Brooks, John'},
          'score': {'after_extra_time': None,
                    'away': 3,
                    'final_score': None,
                    'half_time_away': 2,
                    'half_time_home': 1,
                    'home': 2,
                    'penalty_shootout': None,
                    'regulation': {'away': 3, 'home': 2},
                    'went_to_extra_time': False,
                    'we

In [327]:
# ---------------------------------------------------------
# TEST PROCESSOR ON ONE 2026/27 RAW MATCH
# ---------------------------------------------------------

sample_file = match_files[0]

print(
    "FILE:",
    sample_file
)

print()

processed_sample = process_match_file(
    sample_file
)

print(
    processed_sample
)

FILE: ..\data\raw\matches\premier_league\2026_27\mt_022587821.json

{'match_id': 'mt_022587821', 'date': '2026-08-24T19:00:00.000Z', 'season': '2026_27', 'competition': 'premier_league', 'matchday': 1, 'home_team': 'Fulham', 'away_team': 'Chelsea', 'home_goals': 2, 'away_goals': 3, 'home_xg': None, 'away_xg': None, 'home_shots': None, 'away_shots': None, 'home_shots_on_target': None, 'away_shots_on_target': None, 'home_possession': None, 'away_possession': None, 'home_passes': None, 'away_passes': None, 'home_corners': None, 'away_corners': None, 'winner': 'A'}


In [328]:
# ---------------------------------------------------------
# PROCESS ALL 20 2026/27 FILES DIRECTLY
# ---------------------------------------------------------

test_2026_rows = []

for file_path in sorted(
    Path("../data/raw/matches/premier_league/2026_27").glob("*.json")
):

    row = process_match_file(
        file_path
    )

    if row is not None:
        test_2026_rows.append(row)


test_2026_df = pd.DataFrame(
    test_2026_rows
)


print(
    "Processed matches:",
    len(test_2026_df)
)

print()

print(
    test_2026_df[
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals",
            "winner"
        ]
    ]
    .sort_values("date")
    .to_string(index=False)
)

print()

print(
    "Missing dates:",
    test_2026_df["date"].isna().sum()
)

print(
    "Missing home goals:",
    test_2026_df["home_goals"].isna().sum()
)

print(
    "Missing away goals:",
    test_2026_df["away_goals"].isna().sum()
)

print(
    "Missing winners:",
    test_2026_df["winner"].isna().sum()
)

Processed matches: 20

                    date              home_team              away_team  home_goals  away_goals winner
2026-08-21T19:00:00.000Z                Arsenal          Coventry City           3           0      H
2026-08-22T11:30:00.000Z              Hull City      Manchester United           2           0      H
2026-08-22T14:00:00.000Z      Nottingham Forest           Leeds United           0           1      A
2026-08-22T14:00:00.000Z           Ipswich Town             Sunderland           2           1      H
2026-08-22T14:00:00.000Z                Everton         Crystal Palace           2           0      H
2026-08-22T16:30:00.000Z              Brentford      Tottenham Hotspur           3           0      H
2026-08-23T13:00:00.000Z Brighton & Hove Albion            Aston Villa           4           0      H
2026-08-23T13:00:00.000Z        Manchester City            Bournemouth           2           1      H
2026-08-23T15:30:00.000Z       Newcastle United            

In [329]:
# ---------------------------------------------------------
# CLEAN REBUILD OF HISTORICAL DATASET
# ---------------------------------------------------------

rows = []


# Process every raw match again from scratch
for file_path in RAW_MATCH_DIR.rglob("*.json"):

    row = process_match_file(
        file_path
    )

    if row is not None:
        rows.append(row)


# Create completely new dataframe
historical_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# REMOVE DUPLICATES
# ---------------------------------------------------------

historical_df = (
    historical_df
    .drop_duplicates(
        subset="match_id",
        keep="last"
    )
    .copy()
)


# ---------------------------------------------------------
# CONVERT AND SORT DATES
# ---------------------------------------------------------

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# SAVE FRESH CSV
# ---------------------------------------------------------

output_path = (
    PROCESSED_DIR
    / "historical_matches.csv"
)


historical_df.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# VERIFY 2026/27
# ---------------------------------------------------------

check_2026 = (
    historical_df[
        (
            historical_df["competition"]
            == "premier_league"
        )
        &
        (
            historical_df["season"]
            == "2026_27"
        )
    ][
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals",
            "winner"
        ]
    ]
    .sort_values("date")
)


print(
    "Historical shape:",
    historical_df.shape
)

print()

print(
    "2026/27 PL matches:",
    len(check_2026)
)

print()

print(
    check_2026.to_string(
        index=False
    )
)

print()

print(
    "Missing dates:",
    check_2026["date"].isna().sum()
)

print(
    "Missing home goals:",
    check_2026["home_goals"].isna().sum()
)

print(
    "Missing away goals:",
    check_2026["away_goals"].isna().sum()
)

print(
    "Missing winners:",
    check_2026["winner"].isna().sum()
)

print()

print(
    "Saved:",
    output_path
)

Historical shape: (10673, 22)

2026/27 PL matches: 20

                     date              home_team              away_team  home_goals  away_goals winner
2026-08-21 19:00:00+00:00                Arsenal          Coventry City           3           0      H
2026-08-22 11:30:00+00:00              Hull City      Manchester United           2           0      H
2026-08-22 14:00:00+00:00                Everton         Crystal Palace           2           0      H
2026-08-22 14:00:00+00:00           Ipswich Town             Sunderland           2           1      H
2026-08-22 14:00:00+00:00      Nottingham Forest           Leeds United           0           1      A
2026-08-22 16:30:00+00:00              Brentford      Tottenham Hotspur           3           0      H
2026-08-23 13:00:00+00:00        Manchester City            Bournemouth           2           1      H
2026-08-23 13:00:00+00:00 Brighton & Hove Albion            Aston Villa           4           0      H
2026-08-23 15:30:0

In [330]:
# ---------------------------------------------------------
# INSPECT RAW STATISTICS STRUCTURE
# ---------------------------------------------------------

from pprint import pprint
import json
from pathlib import Path


sample_file = Path(
    "../data/raw/matches/premier_league/2026_27/mt_022587821.json"
)


with open(
    sample_file,
    "r",
    encoding="utf-8"
) as f:

    sample_match = json.load(f)


print("STATISTICS:")
print()

pprint(
    sample_match.get("statistics")
)

STATISTICS:

{'data': {'attack': {'big_chances_missed': {'all': {'away': 1, 'home': 1},
                                            'first_half': {'away': 1,
                                                           'home': 0},
                                            'second_half': {'away': 0,
                                                            'home': 1}},
                     'fouled_in_final_third': {'all': {'away': 1, 'home': 0},
                                               'first_half': {'away': 1,
                                                              'home': 0},
                                               'second_half': {'away': 0,
                                                               'home': 0}},
                     'offsides': {'all': {'away': 3, 'home': 0},
                                  'first_half': {'away': 1, 'home': 0},
                                  'second_half': {'away': 2, 'home': 0}},
                     'touches_in_penalty_

In [331]:
# ---------------------------------------------------------
# SAFE STAT EXTRACTOR
# ---------------------------------------------------------

def get_stat(statistics, team_side, stat_name):

    if statistics is None:
        return None


    # Unwrap API "data"
    if isinstance(statistics, dict):

        statistics = statistics.get(
            "data",
            statistics
        )


    if not isinstance(statistics, dict):
        return None


    # -----------------------------------------------------
    # MAP OUR DATASET NAMES TO API STAT NAMES
    # -----------------------------------------------------

    stat_name_map = {

        "expected_goals":
            "expected_goals",

        "shots":
            "total_shots",

        "shots_on_target":
            "shots_on_target",

        "ball_possession":
            "ball_possession",

        "passes":
            "passes",

        "corners":
            "corner_kicks"
    }


    api_stat_name = stat_name_map.get(
        stat_name,
        stat_name
    )


    # -----------------------------------------------------
    # NEW API STRUCTURE:
    # data -> overview -> stat -> all -> home/away
    # -----------------------------------------------------

    overview = statistics.get(
        "overview",
        {}
    )


    if isinstance(overview, dict):

        stat_data = overview.get(
            api_stat_name
        )


        if isinstance(stat_data, dict):

            all_data = stat_data.get(
                "all"
            )


            if isinstance(all_data, dict):

                value = all_data.get(
                    team_side
                )


                if value is not None:
                    return value


    # -----------------------------------------------------
    # FALLBACK FOR OLDER RAW FILE STRUCTURES
    # -----------------------------------------------------

    team_stats = statistics.get(
        team_side,
        {}
    )


    if isinstance(team_stats, dict):

        value = team_stats.get(
            stat_name
        )


        if value is not None:
            return value


    return None

In [332]:
# ---------------------------------------------------------
# TEST UPDATED STAT EXTRACTION
# ---------------------------------------------------------

processed_sample = process_match_file(
    sample_file
)


for column in [
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners"
]:

    print(
        column,
        processed_sample[column]
    )

home_xg 1.33
away_xg 2.24
home_shots 14
away_shots 18
home_shots_on_target 6
away_shots_on_target 6
home_possession 62
away_possession 38
home_passes 639
away_passes 394
home_corners 6
away_corners 4


In [333]:
# ---------------------------------------------------------
# FINAL CLEAN REBUILD WITH CORRECT STAT EXTRACTION
# ---------------------------------------------------------

rows = []


for file_path in RAW_MATCH_DIR.rglob("*.json"):

    row = process_match_file(
        file_path
    )

    if row is not None:
        rows.append(row)


historical_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# REMOVE DUPLICATES
# ---------------------------------------------------------

historical_df = (
    historical_df
    .drop_duplicates(
        subset="match_id",
        keep="last"
    )
    .copy()
)


# ---------------------------------------------------------
# DATE + SORT
# ---------------------------------------------------------

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

output_path = (
    PROCESSED_DIR
    / "historical_matches.csv"
)


historical_df.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# CHECK 2026/27 STAT COVERAGE
# ---------------------------------------------------------

check_2026 = historical_df[
    (
        historical_df["competition"]
        == "premier_league"
    )
    &
    (
        historical_df["season"]
        == "2026_27"
    )
].copy()


print(
    "Historical shape:",
    historical_df.shape
)

print(
    "2026/27 PL matches:",
    len(check_2026)
)

print()

print("2026/27 MISSING VALUES")

for column in [
    "date",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners"
]:

    print(
        f"{column}:",
        check_2026[column].isna().sum()
    )


print()

print(
    check_2026[
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals",
            "home_shots_on_target",
            "away_shots_on_target"
        ]
    ]
    .sort_values("date")
    .to_string(index=False)
)


print()

print(
    "Saved:",
    output_path
)

Historical shape: (10673, 22)
2026/27 PL matches: 20

2026/27 MISSING VALUES
date: 0
home_goals: 0
away_goals: 0
home_xg: 0
away_xg: 0
home_shots: 0
away_shots: 0
home_shots_on_target: 0
away_shots_on_target: 0
home_possession: 0
away_possession: 0
home_passes: 0
away_passes: 0
home_corners: 0
away_corners: 0

                     date              home_team              away_team  home_goals  away_goals  home_shots_on_target  away_shots_on_target
2026-08-21 19:00:00+00:00                Arsenal          Coventry City           3           0                   6.0                   1.0
2026-08-22 11:30:00+00:00              Hull City      Manchester United           2           0                   4.0                   5.0
2026-08-22 14:00:00+00:00                Everton         Crystal Palace           2           0                   5.0                   4.0
2026-08-22 14:00:00+00:00           Ipswich Town             Sunderland           2           1                   3.0           